# Investment Research System — Shared Infrastructure

This notebook builds the **shared infrastructure layer** used by all future agents in the
multi-agent investment research system. **No agents are defined here** — only environment
setup, reusable tools (Tavily + Financial Modeling Prep), shared Pydantic data models,
logging, and the output directory structure.

> Runs in **Google Colab** or **locally**. In Colab, API keys are read from **Colab Secrets**
> (the key icon in the left sidebar); locally they fall back to a `.env` file. Run all cells
> top-to-bottom — Section 1 installs dependencies, loads & validates keys, creates the output
> directories, and runs a connectivity smoke test.

**Notebook layout**
1. Environment Setup
2. Tavily Tool
3. Financial Modeling Prep Tools
4. Data Models
5. Logging Utilities
6. Investment Mandate
7. Infrastructure Validation Tests
8. Agent — Fixed Income Researcher
9. Agent — ETF Researcher
10. Agent — Equity Researcher
11. Agent — Portfolio Constructor
12. Agent — Investment Committee Reviewer


## Section 1 — Environment Setup

Colab-compatible bootstrap. This section:
1. Installs the required dependencies (`%pip`).
2. Loads API keys from **Colab Secrets** when running in Colab, falling back to a local **`.env`** file.
3. Validates that `OPENAI_API_KEY`, `TAVILY_API_KEY` and `FMP_API_KEY` are present.
4. Creates the output directory structure.
5. Runs a connectivity smoke test against OpenAI, Tavily and FMP.

In [1]:
# Dependency installation (safe to run in Colab or locally; "-q" keeps it quiet).
%pip install -q openai-agents python-dotenv requests

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import json
import datetime as dt
from pathlib import Path
from typing import TypedDict, Optional

import requests
from dotenv import load_dotenv
from pydantic import BaseModel, Field

# Detect Google Colab.
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# Locally, populate os.environ from a .env file (no-op in Colab if absent).
load_dotenv()


def get_secret(name: str) -> Optional[str]:
    """Resolve a secret from Colab Secrets first (when in Colab), else the environment/.env."""
    if IN_COLAB:
        try:
            value = userdata.get(name)
            if value:
                return value
        except Exception:
            pass  # secret not set / access not granted -> fall through to env
    return os.getenv(name)


REQUIRED_KEYS = ["OPENAI_API_KEY", "TAVILY_API_KEY", "FMP_API_KEY"]

# Load each key and mirror it into os.environ so the OpenAI Agents SDK (which reads
# OPENAI_API_KEY from the environment) works the same in Colab and locally.
_secrets = {k: get_secret(k) for k in REQUIRED_KEYS}
for _k, _v in _secrets.items():
    if _v:
        os.environ[_k] = _v

OPENAI_API_KEY = _secrets["OPENAI_API_KEY"]
TAVILY_API_KEY = _secrets["TAVILY_API_KEY"]
FMP_API_KEY = _secrets["FMP_API_KEY"]

print(f"Environment: {'Google Colab' if IN_COLAB else 'local'} "
      f"({'Colab Secrets' if IN_COLAB else '.env'} as primary key source)")

Environment: local (.env as primary key source)


In [3]:
# Validate that all required API keys are present.
missing = [k for k in REQUIRED_KEYS if not _secrets.get(k)]
if missing:
    source = "Colab Secrets (sidebar key icon)" if IN_COLAB else "your .env file"
    raise EnvironmentError(
        f"Missing required API key(s): {missing}. Add them to {source} and re-run this section."
    )

print("All required API keys loaded:")
for k in REQUIRED_KEYS:
    v = _secrets[k]
    masked = f"{v[:6]}...{v[-4:]}" if v and len(v) > 12 else "set"
    print(f"  - {k:16s} {masked}")

All required API keys loaded:
  - OPENAI_API_KEY   sk-pro...MxgA
  - TAVILY_API_KEY   tvly-d...AcN0
  - FMP_API_KEY      uCiRiX...YxPX


In [4]:
# Output directory structure -- created automatically and idempotently.
BASE_DIR = Path.cwd()
OUTPUT_DIRS = {
    "outputs":    BASE_DIR / "outputs",
    "research":   BASE_DIR / "outputs" / "research",
    "portfolios": BASE_DIR / "outputs" / "portfolios",
    "reports":    BASE_DIR / "outputs" / "reports",
    "logs":       BASE_DIR / "outputs" / "logs",
}
for path in OUTPUT_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

print("Output directories ready:")
for name, path in OUTPUT_DIRS.items():
    print(f"  - {name:10s} -> {path}")

Output directories ready:
  - outputs    -> /Users/ogurianova/Desktop/ANTON TASK1/outputs
  - research   -> /Users/ogurianova/Desktop/ANTON TASK1/outputs/research
  - portfolios -> /Users/ogurianova/Desktop/ANTON TASK1/outputs/portfolios
  - reports    -> /Users/ogurianova/Desktop/ANTON TASK1/outputs/reports
  - logs       -> /Users/ogurianova/Desktop/ANTON TASK1/outputs/logs


In [5]:
# Connectivity smoke test -- verifies each external service responds with the loaded keys.
def _smoke_openai() -> str:
    r = requests.get(
        "https://api.openai.com/v1/models",
        headers={"Authorization": f"Bearer {OPENAI_API_KEY}"}, timeout=30,
    )
    r.raise_for_status()
    return f"OpenAI reachable — {len(r.json().get('data', []))} models visible"


def _smoke_tavily() -> str:
    r = requests.post(
        "https://api.tavily.com/search",
        json={"api_key": TAVILY_API_KEY, "query": "connectivity test", "max_results": 1},
        timeout=30,
    )
    r.raise_for_status()
    return f"Tavily reachable — {len(r.json().get('results', []))} result(s)"


def _smoke_fmp() -> str:
    r = requests.get(
        "https://financialmodelingprep.com/stable/quote",
        params={"symbol": "AAPL", "apikey": FMP_API_KEY}, timeout=30,
    )
    r.raise_for_status()
    data = r.json()
    price = data[0].get("price") if isinstance(data, list) and data else None
    return f"FMP reachable — AAPL price {price}"


print("Connectivity smoke test:")
_all_ok = True
for _name, _fn in [("OpenAI", _smoke_openai), ("Tavily", _smoke_tavily), ("FMP", _smoke_fmp)]:
    try:
        print("  [OK]  ", _fn())
    except Exception as e:
        _all_ok = False
        print(f"  [FAIL] {_name}: {e}")
print("\nAll services reachable." if _all_ok else "\nOne or more services failed — check the keys above.")

Connectivity smoke test:
  [OK]   OpenAI reachable — 118 models visible
  [OK]   Tavily reachable — 1 result(s)
  [OK]   FMP reachable — AAPL price 292.43

All services reachable.


## Section 2 — Tavily Web Search Tool

A typed wrapper around the Tavily search API (`https://api.tavily.com/search`). Returns a list
of **normalized** result dicts (`title`, `url`, `content`, `score`) so downstream agents get a
stable shape regardless of API changes.

In [6]:
class TavilySearchInput(TypedDict):
    query: str
    max_results: int


TAVILY_SEARCH_URL = "https://api.tavily.com/search"


def tavily_search(query: str, max_results: int = 5) -> list[dict]:
    """Run a web search via the Tavily API and return normalized results.

    Args:
        query: Natural-language search query.
        max_results: Maximum number of results to return (default 5).

    Returns:
        A list of dicts, each with keys: ``title``, ``url``, ``content``, ``score``.
    """
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results,
        "search_depth": "advanced",
        "include_answer": False,
    }
    resp = requests.post(TAVILY_SEARCH_URL, json=payload, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    return [
        {
            "title": r.get("title", ""),
            "url": r.get("url", ""),
            "content": r.get("content", ""),
            "score": r.get("score"),
        }
        for r in data.get("results", [])
    ]

## Section 3 — Financial Modeling Prep Tools

Wrappers around the FMP **`stable`** API (the legacy `/api/v3` endpoints were retired on
2025-08-31 and are unavailable for current keys).

- `fmp_quote` — latest price quote
- `fmp_company_profile` — company fundamentals / profile
- `fmp_etf_profile` — ETF profile (tries the dedicated `etf/info` endpoint, falls back to the
  general `profile` endpoint, which works for ETFs on all plans)
- `fmp_dividend_info` — dividend history + summary

A shared `_fmp_get` helper handles auth, error payloads, and restricted-endpoint text responses.

In [7]:
FMP_BASE_URL = "https://financialmodelingprep.com/stable"


def _fmp_get(endpoint: str, params: Optional[dict] = None):
    """Low-level GET against the FMP `stable` API. Returns parsed JSON.

    Raises:
        RuntimeError: if FMP returns a restricted/legacy/error payload.
    """
    params = dict(params or {})
    params["apikey"] = FMP_API_KEY
    resp = requests.get(f"{FMP_BASE_URL}/{endpoint}", params=params, timeout=30)
    resp.raise_for_status()

    text = resp.text.strip()
    if text.startswith(("Restricted Endpoint", "Legacy Endpoint")):
        raise RuntimeError(text)

    data = resp.json()
    if isinstance(data, dict) and ("Error Message" in data or "Error" in data):
        raise RuntimeError(data.get("Error Message") or data.get("Error"))
    return data


def fmp_quote(symbol: str) -> dict:
    """Latest price quote for a stock or ETF symbol."""
    data = _fmp_get("quote", {"symbol": symbol.upper()})
    return data[0] if isinstance(data, list) and data else {}


def fmp_company_profile(symbol: str) -> dict:
    """Company profile / fundamentals for a symbol."""
    data = _fmp_get("profile", {"symbol": symbol.upper()})
    return data[0] if isinstance(data, list) and data else {}


def fmp_etf_profile(symbol: str) -> dict:
    """ETF profile.

    Tries the dedicated ``etf/info`` endpoint first; if it is restricted under the current
    plan, falls back to the general ``profile`` endpoint (which returns ETF data too).
    """
    symbol = symbol.upper()
    try:
        data = _fmp_get("etf/info", {"symbol": symbol})
        if isinstance(data, list) and data:
            return {**data[0], "_source": "etf/info"}
        if isinstance(data, dict) and data:
            return {**data, "_source": "etf/info"}
    except (RuntimeError, requests.HTTPError):
        pass  # endpoint restricted on this plan -> fall back
    profile = fmp_company_profile(symbol)
    profile["_source"] = "profile_fallback"
    return profile


def fmp_dividend_info(symbol: str) -> dict:
    """Dividend history plus a small summary for a symbol."""
    data = _fmp_get("dividends", {"symbol": symbol.upper()})
    history = data if isinstance(data, list) else []
    latest = history[0] if history else {}
    return {
        "symbol": symbol.upper(),
        "latest_dividend": latest.get("dividend"),
        "latest_date": latest.get("date"),
        "latest_yield": latest.get("yield"),
        "frequency": latest.get("frequency"),
        "history_count": len(history),
        "history": history,
    }

## Section 4 — Shared Data Models

Pydantic models that downstream agents populate and pass between each other. All security-level
candidates carry a free-text `rationale` and `source_urls` so research is traceable. A
`PortfolioCandidate` aggregates the security candidates plus an allocation map.

In [8]:
class BondCandidate(BaseModel):
    """A fixed-income instrument under consideration."""
    symbol: Optional[str] = None
    issuer: str
    bond_type: Optional[str] = None          # e.g. government, corporate, municipal
    coupon_rate: Optional[float] = None       # %
    yield_to_maturity: Optional[float] = None # %
    maturity_date: Optional[str] = None       # ISO date
    credit_rating: Optional[str] = None       # e.g. AAA, BBB+
    face_value: Optional[float] = None
    price: Optional[float] = None
    currency: str = "USD"
    rationale: Optional[str] = None
    source_urls: list[str] = Field(default_factory=list)


class EquityCandidate(BaseModel):
    """A single stock under consideration."""
    symbol: str
    company_name: Optional[str] = None
    sector: Optional[str] = None
    industry: Optional[str] = None
    price: Optional[float] = None
    market_cap: Optional[float] = None
    pe_ratio: Optional[float] = None
    dividend_yield: Optional[float] = None    # %
    beta: Optional[float] = None
    currency: str = "USD"
    rationale: Optional[str] = None
    source_urls: list[str] = Field(default_factory=list)


class ETFCandidate(BaseModel):
    """An exchange-traded fund under consideration."""
    symbol: str
    name: Optional[str] = None
    asset_class: Optional[str] = None
    expense_ratio: Optional[float] = None     # %
    aum: Optional[float] = None               # assets under management
    price: Optional[float] = None
    dividend_yield: Optional[float] = None    # %
    holdings_count: Optional[int] = None
    issuer: Optional[str] = None
    currency: str = "USD"
    rationale: Optional[str] = None
    source_urls: list[str] = Field(default_factory=list)


class PortfolioCandidate(BaseModel):
    """A proposed portfolio aggregating multiple security candidates."""
    name: str
    objective: Optional[str] = None
    risk_profile: Optional[str] = None        # conservative, balanced, aggressive
    equities: list[EquityCandidate] = Field(default_factory=list)
    bonds: list[BondCandidate] = Field(default_factory=list)
    etfs: list[ETFCandidate] = Field(default_factory=list)
    allocation: dict[str, float] = Field(default_factory=dict)  # asset class -> weight %
    expected_return: Optional[float] = None   # %
    notes: Optional[str] = None

## Section 5 — Logging Utilities

A simple activity logger that records **timestamp, agent name, query, and result count** for
every research action. Entries are printed to the console and appended as JSON lines to
`outputs/logs/research_activity.log`.

In [9]:
import logging

LOG_FILE = OUTPUT_DIRS["logs"] / "research_activity.log"

_logger = logging.getLogger("investment_research")
_logger.setLevel(logging.INFO)
if not _logger.handlers:  # avoid duplicate handlers on re-run
    _file_handler = logging.FileHandler(LOG_FILE, encoding="utf-8")
    _file_handler.setFormatter(logging.Formatter("%(message)s"))
    _logger.addHandler(_file_handler)
    _stream_handler = logging.StreamHandler()
    _stream_handler.setFormatter(logging.Formatter("%(message)s"))
    _logger.addHandler(_stream_handler)


def log_activity(agent_name: str, query: str, result_count: int) -> dict:
    """Record a research activity entry.

    Captures ``timestamp``, ``agent``, ``query`` and ``result_count``; logs to console and
    appends one JSON line to ``outputs/logs/research_activity.log``. Returns the entry dict.
    """
    entry = {
        "timestamp": dt.datetime.now().isoformat(timespec="seconds"),
        "agent": agent_name,
        "query": query,
        "result_count": result_count,
    }
    _logger.info(json.dumps(entry))
    return entry

## Section 6 — Investment Mandate

The **investment mandate** is the shared set of constraints every downstream agent must respect
(portfolio size, target yield, rating floors, allocations, preferred sectors, etc.). It is
persisted to `outputs/context/investment_mandate.json` so agents can load a single source of
truth instead of hard-coding rules.

- `save_investment_mandate(mandate)` — write the mandate to disk (creates `outputs/context/`)
- `load_investment_mandate()` — read it back

In [10]:
MANDATE_PATH = OUTPUT_DIRS["outputs"] / "context" / "investment_mandate.json"


def save_investment_mandate(mandate: dict, path: Path = MANDATE_PATH) -> Path:
    """Persist the investment mandate to outputs/context/investment_mandate.json.

    Creates the ``outputs/context/`` directory if needed. Returns the written path.
    """
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(mandate, f, indent=2, ensure_ascii=False)
    return path


def load_investment_mandate(path: Path = MANDATE_PATH) -> dict:
    """Load the investment mandate from outputs/context/investment_mandate.json."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

In [11]:
# Define and persist the mandate that governs the whole research system.
INVESTMENT_MANDATE = {
    "portfolio_size": "RSD 100M",
    "target_yield": "7% net",
    "minimum_bond_rating": "B-",
    "preferred_bond_rating": "B+",
    "currencies": ["EUR", "USD", "RSD"],
    "regions": ["EU", "USA", "LATAM"],
    "horizon": "3+ years",
    "liquidity": "Liquid instruments only",
    "fixed_income_allocation": "50-65%",
    "equity_allocation": "15-25%",
    "preferred_sectors": [
        "IT Services",
        "Outsourcing",
        "Digital Transformation",
        "Chemicals",
        "Materials",
        "Healthcare",
    ],
}

saved_path = save_investment_mandate(INVESTMENT_MANDATE)
print(f"Investment mandate saved to: {saved_path}")

Investment mandate saved to: /Users/ogurianova/Desktop/ANTON TASK1/outputs/context/investment_mandate.json


In [12]:
mandate = load_investment_mandate()
print(mandate)

{'portfolio_size': 'RSD 100M', 'target_yield': '7% net', 'minimum_bond_rating': 'B-', 'preferred_bond_rating': 'B+', 'currencies': ['EUR', 'USD', 'RSD'], 'regions': ['EU', 'USA', 'LATAM'], 'horizon': '3+ years', 'liquidity': 'Liquid instruments only', 'fixed_income_allocation': '50-65%', 'equity_allocation': '15-25%', 'preferred_sectors': ['IT Services', 'Outsourcing', 'Digital Transformation', 'Chemicals', 'Materials', 'Healthcare']}


## Section 7 — Infrastructure Validation Tests

Live checks confirming the full infrastructure works end-to-end: a sample Tavily search, sample
FMP requests (quote / profile / dividend / ETF), and a Pydantic model round-trip.

In [13]:
print("=" * 64)
print("INFRASTRUCTURE VALIDATION")
print("=" * 64)

# [1] Tavily sample search
print("\n[1] Tavily search ...")
tav_results = tavily_search("US Treasury bond yields 2026 outlook", max_results=3)
log_activity("infra_test", "US Treasury bond yields 2026 outlook", len(tav_results))
print(f"    -> {len(tav_results)} results")
for r in tav_results:
    print(f"       - {r['title'][:68]}  ({r['url']})")

# [2] FMP quote
print("\n[2] FMP quote (AAPL) ...")
quote = fmp_quote("AAPL")
print(f"    -> {quote.get('name')} | price={quote.get('price')} | mktcap={quote.get('marketCap')}")

# [3] FMP profile / dividend / ETF
print("\n[3] FMP profile / dividend / ETF ...")
profile = fmp_company_profile("AAPL")
print(f"    profile : {profile.get('companyName')} | sector={profile.get('sector')}")
dividend = fmp_dividend_info("AAPL")
print(f"    dividend: latest={dividend['latest_dividend']} on {dividend['latest_date']} "
      f"| {dividend['history_count']} records")
etf = fmp_etf_profile("SPY")
print(f"    etf     : {etf.get('companyName') or etf.get('name')} | price={etf.get('price')} "
      f"| source={etf.get('_source')}")

# [4] Pydantic model round-trip
print("\n[4] Data model round-trip ...")
equity = EquityCandidate(
    symbol="AAPL",
    company_name=profile.get("companyName"),
    sector=profile.get("sector"),
    price=quote.get("price"),
    market_cap=quote.get("marketCap"),
    beta=profile.get("beta"),
    rationale="Sample candidate built during infra validation.",
)
portfolio = PortfolioCandidate(
    name="Validation Sample Portfolio",
    risk_profile="balanced",
    equities=[equity],
    allocation={"equity": 60.0, "bond": 30.0, "cash": 10.0},
)
print("    EquityCandidate OK:", equity.symbol, "| PortfolioCandidate OK:", portfolio.name)

print("\nAll infrastructure checks passed.")

INFRASTRUCTURE VALIDATION

[1] Tavily search ...


{"timestamp": "2026-06-12T15:45:26", "agent": "infra_test", "query": "US Treasury bond yields 2026 outlook", "result_count": 3}


    -> 3 results
       - 2026 Mid-Year Outlook: Taxable Fixed Income - Charles Schwab  (https://www.schwab.com/learn/story/fixed-income-outlook)
       - Treasury Yields Snapshot: June 5, 2026 - Advisor Perspectives  (https://www.advisorperspectives.com/dshort/updates/2026/06/05/treasury-yields-snapshot-june-5-2026)
       - 5 Realistic Surprise Predictions for 2026  (https://am.jpmorgan.com/us/en/asset-management/adv/insights/portfolio-insights/fixed-income/5-realistic-surprise-predictions-for-2026)

[2] FMP quote (AAPL) ...
    -> Apple Inc. | price=292.87 | mktcap=4301485951720

[3] FMP profile / dividend / ETF ...
    profile : Apple Inc. | sector=Technology
    dividend: latest=0.27 on 2026-05-11 | 91 records
    etf     : State Street SPDR S&P 500 ETF Trust | price=736.5 | source=profile_fallback

[4] Data model round-trip ...
    EquityCandidate OK: AAPL | PortfolioCandidate OK: Validation Sample Portfolio

All infrastructure checks passed.


## Section 8 — Agent: Fixed Income Researcher

The first agent of the system, built on the OpenAI Agents SDK (`gpt-5-mini`). It:

1. **Loads the mandate** from `outputs/context/investment_mandate.json`.
2. **Researches** with Tavily (corporate bonds, government bonds, bond ETFs, money-market ETFs).
3. **Enriches / validates** candidates with FMP (quotes, profiles, dividends).
4. Returns **30–50 candidates** with rich, instrument-appropriate detail:
   - **Bonds:** ISIN, coupon (distinct from YTM), yield-to-maturity, maturity date, duration, credit rating.
   - **ETFs:** average credit rating, SEC yield, expense ratio.
   - **All:** a **1–100 score** (yield, credit quality, liquidity, mandate fit) and a
     `contributes_to_target_yield` flag marking candidates that do **not** help reach the 7% net target.

Output is a typed `FixedIncomeResearchResult`; results are saved to
`outputs/research/fixed_income_candidates.json`.

In [14]:
from agents import Agent, Runner, function_tool, AgentOutputSchema, ModelSettings


# --- Expose the shared infra functions to the agent as tools ---
@function_tool
def web_search(query: str, max_results: int = 5) -> list[dict]:
    """Search the web (Tavily) for fixed-income instruments, yields, ratings, issuers, ETFs."""
    results = tavily_search(query, max_results=max_results)
    log_activity("FixedIncomeResearcher", query, len(results))
    return results


@function_tool
def fmp_get_quote(symbol: str) -> dict:
    """FMP: latest market quote for a ticker (bond ETF / money-market ETF / equity)."""
    return fmp_quote(symbol)


@function_tool
def fmp_get_profile(symbol: str) -> dict:
    """FMP: issuer / fund profile for a ticker (sector, exchange, description, currency)."""
    return fmp_company_profile(symbol)


@function_tool
def fmp_get_etf_profile(symbol: str) -> dict:
    """FMP: ETF profile for a bond or money-market ETF ticker."""
    return fmp_etf_profile(symbol)


@function_tool
def fmp_get_dividend_info(symbol: str) -> dict:
    """FMP: dividend / distribution history for an ETF or bond fund (proxy for yield)."""
    return fmp_dividend_info(symbol)

In [16]:
from pydantic import field_validator


class FixedIncomeCandidate(BaseModel):
    """A single fixed-income opportunity with instrument-appropriate detail + scoring."""
    # --- identity ---
    name: str                              # full instrument / fund name
    ticker: Optional[str] = None           # exchange ticker (None if not listed)
    isin: Optional[str] = None             # ISIN, when identifiable
    type: str                              # Corporate Bond | Government Bond | Bond ETF | Money Market ETF
    region: Optional[str] = None           # EU | USA | LATAM
    currency: Optional[str] = None         # EUR | USD | RSD

    # --- bond-specific ---
    rating: Optional[str] = None           # instrument/issuer credit rating, e.g. BBB+, B-
    coupon: Optional[str] = None           # stated coupon rate, e.g. "5.25%" (distinct from YTM)
    ytm: Optional[str] = None              # yield to maturity, e.g. "7.1%"
    maturity_date: Optional[str] = None    # ISO date or year, e.g. "2030-06-15"
    duration: Optional[str] = None         # modified/effective duration in years, if available

    # --- ETF-specific ---
    avg_credit_rating: Optional[str] = None  # fund average credit quality, e.g. "BB"
    sec_yield: Optional[str] = None          # 30-day SEC yield, e.g. "6.59%"
    expense_ratio: Optional[str] = None      # e.g. "0.49%"

    # --- scoring & mandate fit ---
    score: Optional[int] = Field(default=None, ge=1, le=100)  # composite 1-100
    score_rationale: Optional[str] = None    # brief justification for the score
    contributes_to_target_yield: Optional[bool] = None  # False = does NOT help reach 7% net target

    # --- provenance ---
    source: Optional[str] = None           # URL / source the candidate was grounded in

    @field_validator("score")
    @classmethod
    def _clamp_score(cls, v):
        if v is None:
            return v
        return max(1, min(100, v))


class FixedIncomeResearchResult(BaseModel):
    """Structured output: the full list of fixed-income candidates."""
    candidates: list[FixedIncomeCandidate]

In [17]:
FIXED_INCOME_INSTRUCTIONS = """\
You are the **Fixed Income Researcher** for an institutional investment desk.

Your job: find **30 to 50** distinct, real fixed-income opportunities that fit the provided
investment mandate, across all four instrument types:
  - Corporate Bonds
  - Government Bonds
  - Bond ETFs
  - Money Market ETFs

Process:
  1. Use `web_search` (Tavily) to discover instruments matching the mandate's currencies,
     regions, rating floor, and target yield. Run several focused searches per instrument type.
  2. For tickers (especially ETFs), use the FMP tools (`fmp_get_quote`, `fmp_get_etf_profile`,
     `fmp_get_profile`, `fmp_get_dividend_info`) to validate existence and enrich data.

Collect instrument-appropriate detail (use null for anything you cannot ground in a tool result):
  - **Individual bonds (corporate & government):** isin, coupon (the stated coupon rate) AND
    ytm (yield to maturity) as SEPARATE fields, maturity_date, duration (if available), rating.
  - **ETFs (bond & money-market):** avg_credit_rating, sec_yield (30-day SEC yield),
    expense_ratio, and ticker/isin.

Scoring — assign every candidate an integer **score from 1 to 100** weighing:
  - Yield (vs the 7% net target),
  - Credit quality (vs the mandate's B- floor / B+ preference),
  - Liquidity (liquid instruments strongly preferred),
  - Overall fit with the mandate (eligible currency, region, horizon).
  Put a one-line justification in `score_rationale`.

Target-yield flag — set `contributes_to_target_yield`:
  - `false` if the instrument's expected net yield is materially below 7% so that it does NOT
    help reach the blended 7% net target on its own (e.g. most money-market / short-term
    government / investment-grade holdings),
  - `true` if its yield is at or above ~7% net and therefore pulls the portfolio toward target.

Grounding rules:
  - Do NOT invent ISINs, tickers, ratings or yields. If unsure, use null rather than guess.
  - Every candidate must trace to a tool result; put the URL in `source`.
  - Aim for diversity across the four types, the three currencies, and the three regions.

Return between 30 and 50 candidates as the structured output.
"""

fixed_income_researcher = Agent(
    name="Fixed Income Researcher",
    model="gpt-5-mini",
    instructions=FIXED_INCOME_INSTRUCTIONS,
    tools=[
        web_search,
        fmp_get_quote,
        fmp_get_profile,
        fmp_get_etf_profile,
        fmp_get_dividend_info,
    ],
    # strict_json_schema=False keeps optional/defaulted fields valid for structured output.
    output_type=AgentOutputSchema(FixedIncomeResearchResult, strict_json_schema=False),
)

FIXED_INCOME_OUTPUT_PATH = OUTPUT_DIRS["research"] / "fixed_income_candidates.json"


async def run_fixed_income_research() -> list[dict]:
    """Run the Fixed Income Researcher against the saved mandate.

    Loads the mandate, runs the agent (Tavily research + FMP enrichment), saves the candidates
    to ``outputs/research/fixed_income_candidates.json`` and returns them as a list of dicts.
    """
    mandate = load_investment_mandate()

    prompt = (
        "Find 30-50 fixed-income investment opportunities that satisfy this mandate.\n\n"
        f"INVESTMENT MANDATE (JSON):\n{json.dumps(mandate, indent=2)}\n\n"
        "Cover corporate bonds, government bonds, bond ETFs, and money market ETFs. "
        "For bonds capture ISIN, coupon, YTM, maturity date, duration and rating; for ETFs "
        "capture average credit rating, SEC yield and expense ratio. Score each 1-100 and flag "
        "whether it contributes to the 7% net target."
    )

    result = await Runner.run(fixed_income_researcher, prompt, max_turns=40)
    candidates = [c.model_dump() for c in result.final_output.candidates]

    with open(FIXED_INCOME_OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(candidates, f, indent=2, ensure_ascii=False)

    log_activity("FixedIncomeResearcher", "run_fixed_income_research", len(candidates))
    print(f"Saved {len(candidates)} candidates to {FIXED_INCOME_OUTPUT_PATH}")
    return candidates

In [18]:
results = await run_fixed_income_research()

print(len(results))
print(results[:3])

{"timestamp": "2026-06-12T15:46:34", "agent": "FixedIncomeResearcher", "query": "Accenture bond 2028 ISIN 2028 coupon Accenture plc bond ISIN 2028", "result_count": 5}
{"timestamp": "2026-06-12T15:46:43", "agent": "FixedIncomeResearcher", "query": "AT&T 4.50% 2043 bond ISIN 'AT&T 2043 ISIN' 'coupon' 'ISIN'", "result_count": 10}
{"timestamp": "2026-06-12T15:46:51", "agent": "FixedIncomeResearcher", "query": "IBM bond ISIN 2031 'ISIN' 'coupon' 'IBM 2031 bond ISIN'", "result_count": 10}
{"timestamp": "2026-06-12T15:47:00", "agent": "FixedIncomeResearcher", "query": "Accenture 3.900% 2027 ISIN '3.900% senior notes due 2027' 'ISIN Accenture 2027'", "result_count": 10}
{"timestamp": "2026-06-12T15:47:10", "agent": "FixedIncomeResearcher", "query": "Infosys USD bond ISIN 2029 'ISIN' 'Infosys 2029 bond ISIN' 'Infosys Ltd bond 2029 ISIN'", "result_count": 10}
{"timestamp": "2026-06-12T15:47:18", "agent": "FixedIncomeResearcher", "query": "SAP bond ISIN 2028 'SAP 2028 ISIN' 'SAP 2.125% 2028 ISIN

Saved 44 candidates to /Users/ogurianova/Desktop/ANTON TASK1/outputs/research/fixed_income_candidates.json
44
[{'name': 'iShares iBoxx $ High Yield Corporate Bond ETF', 'ticker': 'HYG', 'isin': 'US4642885135', 'type': 'Bond ETF', 'region': 'USA', 'currency': 'USD', 'rating': None, 'coupon': None, 'ytm': None, 'maturity_date': None, 'duration': None, 'avg_credit_rating': None, 'sec_yield': None, 'expense_ratio': None, 'score': 78, 'score_rationale': 'High-yield exposure, very liquid — strong contributor to 7% target despite credit risk.', 'contributes_to_target_yield': True, 'source': 'https://www.ishares.com/us/products/239565/ishares-iboxx-high-yield-corporate-bond-etf'}, {'name': 'SPDR Bloomberg High Yield Bond ETF', 'ticker': 'JNK', 'isin': 'US78468R6229', 'type': 'Bond ETF', 'region': 'USA', 'currency': 'USD', 'rating': None, 'coupon': None, 'ytm': None, 'maturity_date': None, 'duration': None, 'avg_credit_rating': None, 'sec_yield': None, 'expense_ratio': None, 'score': 82, 'score

## Section 9 — Agent: ETF Researcher

The second research agent (OpenAI Agents SDK, `gpt-5-mini`). It researches **ETFs** that fit the
mandate, reusing the shared Tavily + FMP `function_tool`s defined in Section 8.

**Categories covered:** Bond, High Yield Bond, Investment Grade Bond, Emerging Market Bond,
Healthcare, Technology / Digital Transformation, Dividend, and Money Market ETFs.

**Per ETF:** name, ticker, ISIN, ETF category, region, currency, SEC yield, expense ratio, AUM,
average credit rating (bond ETFs), and source — plus a **1–100 score** ranking by mandate fit,
yield, liquidity, and expense ratio. Saved to `outputs/research/etf_candidates.json`.

In [19]:
class ETFResearchCandidate(BaseModel):
    """A single ETF opportunity with mandate-relevant detail + ranking score."""
    name: str                              # full fund name
    ticker: Optional[str] = None           # exchange ticker
    isin: Optional[str] = None             # ISIN, if available
    etf_category: str                      # Bond | High Yield Bond | Investment Grade Bond |
                                           # Emerging Market Bond | Healthcare |
                                           # Technology / Digital Transformation | Dividend | Money Market
    region: Optional[str] = None           # EU | USA | LATAM
    currency: Optional[str] = None         # EUR | USD | RSD
    sec_yield: Optional[str] = None        # 30-day SEC yield, e.g. "6.59%"
    expense_ratio: Optional[str] = None    # e.g. "0.49%"
    aum: Optional[str] = None              # assets under management, e.g. "$18.2B"
    avg_credit_rating: Optional[str] = None  # for bond ETFs, e.g. "BB"
    score: Optional[int] = Field(default=None, ge=1, le=100)  # ranking score 1-100
    score_rationale: Optional[str] = None  # brief justification
    source: Optional[str] = None           # URL the candidate was grounded in

    @field_validator("score")
    @classmethod
    def _clamp_score(cls, v):
        if v is None:
            return v
        return max(1, min(100, v))


class ETFResearchResult(BaseModel):
    """Structured output: the full list of ETF candidates."""
    candidates: list[ETFResearchCandidate]

In [20]:
ETF_RESEARCH_INSTRUCTIONS = """\
You are the **ETF Researcher** for an institutional investment desk.

Your job: find real, liquid ETFs that fit the provided investment mandate. Cover ALL of these
categories (aim for several quality ETFs in each):
  - Bond ETFs
  - High Yield Bond ETFs
  - Investment Grade Bond ETFs
  - Emerging Market Bond ETFs
  - Healthcare ETFs
  - Technology / Digital Transformation ETFs
  - Dividend ETFs
  - Money Market ETFs

Process:
  1. Use `web_search` (Tavily) to discover ETFs matching the mandate's currencies, regions and
     preferred sectors. Run focused searches per category.
  2. For each ticker, use the FMP tools (`fmp_get_etf_profile`, `fmp_get_quote`, `fmp_get_profile`,
     `fmp_get_dividend_info`) to validate the fund exists and enrich its data.

Collect (use null for anything you cannot ground in a tool result):
  name, ticker, isin, etf_category (one of the categories above), region, currency,
  sec_yield, expense_ratio, aum, avg_credit_rating (bond ETFs only), source.

Scoring — assign every ETF an integer **score from 1 to 100**, ranking by:
  - Fit with the mandate (eligible currency/region, preferred sectors, horizon),
  - Yield (vs the 7% net target),
  - Liquidity (large AUM / tight trading strongly preferred),
  - Expense ratio (lower is better).
  Put a one-line justification in `score_rationale`.

Grounding rules:
  - Do NOT invent ISINs, tickers, yields or ratings. If unsure, use null rather than guess.
  - Every candidate must trace to a tool result; put the URL in `source`.
  - Prefer eligible currencies (EUR, USD, RSD) and regions (EU, USA, LATAM).

Return a diverse set spanning all categories as the structured output.
"""

etf_researcher = Agent(
    name="ETF Researcher",
    model="gpt-5-mini",
    instructions=ETF_RESEARCH_INSTRUCTIONS,
    tools=[
        web_search,
        fmp_get_quote,
        fmp_get_profile,
        fmp_get_etf_profile,
        fmp_get_dividend_info,
    ],
    output_type=AgentOutputSchema(ETFResearchResult, strict_json_schema=False),
)

ETF_OUTPUT_PATH = OUTPUT_DIRS["research"] / "etf_candidates.json"


async def run_etf_research() -> list[dict]:
    """Run the ETF Researcher against the saved mandate.

    Loads the mandate, runs the agent (Tavily research + FMP enrichment), ranks ETFs by mandate
    fit / yield / liquidity / expense ratio, saves to ``outputs/research/etf_candidates.json``
    and returns the candidates as a list of dicts (sorted by score, highest first).
    """
    mandate = load_investment_mandate()

    prompt = (
        "Research ETFs that satisfy this mandate.\n\n"
        f"INVESTMENT MANDATE (JSON):\n{json.dumps(mandate, indent=2)}\n\n"
        "Cover bond, high yield bond, investment grade bond, emerging market bond, healthcare, "
        "technology / digital transformation, dividend and money market ETFs. For each ETF collect "
        "name, ticker, ISIN, category, region, currency, SEC yield, expense ratio, AUM, average "
        "credit rating (bond ETFs) and source. Score each 1-100 by mandate fit, yield, liquidity "
        "and expense ratio."
    )

    result = await Runner.run(etf_researcher, prompt, max_turns=40)
    candidates = [c.model_dump() for c in result.final_output.candidates]
    candidates.sort(key=lambda c: (c.get("score") is not None, c.get("score") or 0), reverse=True)

    with open(ETF_OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(candidates, f, indent=2, ensure_ascii=False)

    log_activity("ETFResearcher", "run_etf_research", len(candidates))
    print(f"Saved {len(candidates)} ETF candidates to {ETF_OUTPUT_PATH}")
    return candidates

In [21]:
results = await run_etf_research()

print(len(results))
print(results[:3])

{"timestamp": "2026-06-12T15:53:10", "agent": "FixedIncomeResearcher", "query": "HYG ETF iShares iBoxx $ High Yield Corporate Bond HYG SEC yield expense ratio AUM", "result_count": 5}
{"timestamp": "2026-06-12T15:53:47", "agent": "FixedIncomeResearcher", "query": "JNK SPDR Bloomberg High Yield Bond ETF expense ratio AUM SEC yield JNK fact sheet", "result_count": 5}
{"timestamp": "2026-06-12T15:53:53", "agent": "FixedIncomeResearcher", "query": "LQD expense ratio SEC yield AUM iShares iBoxx Investment Grade Corporate Bond ETF fact sheet 30 day SEC Yield AUM", "result_count": 5}
{"timestamp": "2026-06-12T15:54:14", "agent": "FixedIncomeResearcher", "query": "VCIT Vanguard Intermediate-Term Corporate Bond ETF expense ratio AUM 30 day SEC yield fact sheet VCIT expense ratio AUM", "result_count": 5}
{"timestamp": "2026-06-12T15:54:21", "agent": "FixedIncomeResearcher", "query": "EMB iShares JP Morgan USD Emerging Markets Bond ETF 30 day SEC yield expense ratio AUM fact sheet EMB fact sheet"

Saved 14 ETF candidates to /Users/ogurianova/Desktop/ANTON TASK1/outputs/research/etf_candidates.json
14
[{'name': 'iShares iBoxx $ High Yield Corporate Bond ETF', 'ticker': 'HYG', 'isin': 'US4642885135', 'etf_category': 'High Yield Bond ETFs', 'region': 'USA', 'currency': 'USD', 'sec_yield': '6.70%', 'expense_ratio': '0.49%', 'aum': '$16,378.81M', 'avg_credit_rating': 'BB', 'score': 88, 'score_rationale': 'Close to 7% target (6.7%), very liquid AUM, fits HY allocation despite moderate expense ratio.', 'source': 'https://www.ishares.com/us/products/239565/ishares-iboxx-high-yield-corporate-bond-etf'}, {'name': 'SPDR Bloomberg High Yield Bond ETF', 'ticker': 'JNK', 'isin': 'US78468R6229', 'etf_category': 'High Yield Bond ETFs', 'region': 'USA', 'currency': 'USD', 'sec_yield': '6.68%', 'expense_ratio': '0.40%', 'aum': '$7,735.55M', 'avg_credit_rating': 'BB', 'score': 85, 'score_rationale': 'High yield near target, large liquidity and slightly lower fees than peers.', 'source': 'https://w

## Section 10 — Agent: Equity Researcher

The third research agent (OpenAI Agents SDK, `gpt-5-mini`). It researches **publicly traded
equities** in the mandate's preferred sectors, reusing the shared Tavily + FMP `function_tool`s.

**Sectors:** IT Services, Outsourcing, Digital Transformation, Chemicals, Materials, Healthcare.

**Per company:** name, ticker, sector, country, market cap, dividend yield, P/E, analyst target
price, 1-year return, a 1–2 sentence investment thesis, and source — plus a **1–100 score**
ranking by mandate fit, growth potential, financial strength, and liquidity. Returns 20–30
companies, saved to `outputs/research/equity_candidates.json`.

In [22]:
class EquityResearchCandidate(BaseModel):
    """A single equity opportunity with mandate-relevant detail + ranking score."""
    company_name: str
    ticker: Optional[str] = None
    sector: Optional[str] = None           # one of the mandate's preferred sectors
    country: Optional[str] = None
    market_cap: Optional[str] = None       # e.g. "$45.2B"
    dividend_yield: Optional[str] = None   # e.g. "2.1%"
    pe_ratio: Optional[str] = None         # trailing or forward P/E
    analyst_target_price: Optional[str] = None  # consensus target, if available
    one_year_return: Optional[str] = None  # trailing 1-year total/price return, if available
    investment_thesis: Optional[str] = None     # 1-2 sentences
    score: Optional[int] = Field(default=None, ge=1, le=100)  # ranking score 1-100
    score_rationale: Optional[str] = None
    source: Optional[str] = None

    @field_validator("score")
    @classmethod
    def _clamp_score(cls, v):
        if v is None:
            return v
        return max(1, min(100, v))


class EquityResearchResult(BaseModel):
    """Structured output: the full list of equity candidates."""
    candidates: list[EquityResearchCandidate]

In [23]:
EQUITY_RESEARCH_INSTRUCTIONS = """\
You are the **Equity Researcher** for an institutional investment desk.

Your job: find **20 to 30** real, publicly traded equities that fit the provided investment
mandate, focused on its preferred sectors:
  - IT Services
  - Outsourcing
  - Digital Transformation
  - Chemicals
  - Materials
  - Healthcare

Aim for a diverse spread across these sectors (several names each where possible).\n\nEfficiency: you have a limited turn budget. Make AT MOST one or two tool calls per company (a single FMP profile/quote call is usually enough to enrich it); if a field cannot be obtained quickly, leave it null and move on. Once you have 20-30 companies, STOP researching and return the structured output.

Process:
  1. Use `web_search` (Tavily) to discover leading public companies in each sector, considering
     the mandate's regions (EU, USA, LATAM) and currencies.
  2. For each ticker, use the FMP tools (`fmp_get_quote`, `fmp_get_profile`, `fmp_get_dividend_info`)
     to validate the company exists and enrich price, market cap, P/E, dividend and sector data.

Collect (use null for anything you cannot ground in a tool result):
  company_name, ticker, sector (one of the mandate sectors), country, market_cap, dividend_yield,
  pe_ratio, analyst_target_price, one_year_return, investment_thesis (1-2 sentences), source.

Scoring — assign every company an integer **score from 1 to 100**, ranking by:
  - Mandate fit (preferred sector, eligible region/currency, horizon),
  - Growth potential,
  - Financial strength (profitability, balance sheet, valuation),
  - Liquidity (large, liquid listings preferred).
  Put a one-line justification in `score_rationale`.

Grounding rules:
  - Do NOT invent tickers, prices or ratios. If unsure, use null rather than guess.
  - Every candidate must trace to a tool result; put the URL in `source`.
  - The investment_thesis must be specific to the company, not generic boilerplate.

Return between 20 and 30 companies as the structured output.
"""

equity_researcher = Agent(
    name="Equity Researcher",
    model="gpt-5-mini",
    instructions=EQUITY_RESEARCH_INSTRUCTIONS,
    tools=[
        web_search,
        fmp_get_quote,
        fmp_get_profile,
        fmp_get_dividend_info,
    ],
    output_type=AgentOutputSchema(EquityResearchResult, strict_json_schema=False),
)

EQUITY_OUTPUT_PATH = OUTPUT_DIRS["research"] / "equity_candidates.json"


async def run_equity_research() -> list[dict]:
    """Run the Equity Researcher against the saved mandate.

    Loads the mandate, runs the agent (Tavily research + FMP enrichment), ranks companies by
    mandate fit / growth / financial strength / liquidity, saves to
    ``outputs/research/equity_candidates.json`` and returns them as a list of dicts
    (sorted by score, highest first).
    """
    mandate = load_investment_mandate()

    prompt = (
        "Research 20-30 publicly traded equities that fit this mandate.\n\n"
        f"INVESTMENT MANDATE (JSON):\n{json.dumps(mandate, indent=2)}\n\n"
        "Focus on the preferred sectors: IT Services, Outsourcing, Digital Transformation, "
        "Chemicals, Materials, Healthcare. For each company collect name, ticker, sector, country, "
        "market cap, dividend yield, P/E, analyst target price, 1-year return, a 1-2 sentence "
        "investment thesis and source. Score each 1-100 by mandate fit, growth potential, "
        "financial strength and liquidity."
    )

    result = await Runner.run(equity_researcher, prompt, max_turns=80)
    candidates = [c.model_dump() for c in result.final_output.candidates]
    candidates.sort(key=lambda c: (c.get("score") is not None, c.get("score") or 0), reverse=True)

    with open(EQUITY_OUTPUT_PATH, "w", encoding="utf-8") as f:
        json.dump(candidates, f, indent=2, ensure_ascii=False)

    log_activity("EquityResearcher", "run_equity_research", len(candidates))
    print(f"Saved {len(candidates)} equity candidates to {EQUITY_OUTPUT_PATH}")
    return candidates

In [24]:
results = await run_equity_research()

print(len(results))
print(results[:3])

{"timestamp": "2026-06-12T15:57:14", "agent": "FixedIncomeResearcher", "query": "top IT services companies Accenture Capgemini Cognizant Reply Sopra Steria Globant Totvs list; top chemicals materials companies BASF Covestro Linde Dow PPG Braskem; top healthcare companies JNJ Abbott Sanofi Bayer Novo Nordisk Medtronic; list across EU USA LATAM", "result_count": 10}
{"timestamp": "2026-06-12T15:58:22", "agent": "EquityResearcher", "query": "run_equity_research", "result_count": 20}


Saved 20 equity candidates to /Users/ogurianova/Desktop/ANTON TASK1/outputs/research/equity_candidates.json
20
[{'company_name': 'Johnson & Johnson', 'ticker': 'JNJ', 'sector': 'Healthcare', 'country': 'US', 'market_cap': '574,698,979,657 USD', 'dividend_yield': '2.20%', 'pe_ratio': None, 'analyst_target_price': None, 'one_year_return': None, 'investment_thesis': 'Diversified healthcare leader with strong pharma and medtech franchises and sizeable free cash flow; defensive exposure with consistent dividend history and innovation pipeline in specialty medicines.', 'score': 95, 'score_rationale': 'Top-tier healthcare fit, highly liquid large cap, strong financials and stable dividend—excellent mandate fit.', 'source': 'https://www.jnj.com'}, {'company_name': 'Novo Nordisk A/S', 'ticker': 'NVO', 'sector': 'Healthcare', 'country': 'DK', 'market_cap': '195,237,546,309 USD', 'dividend_yield': '4.10%', 'pe_ratio': None, 'analyst_target_price': None, 'one_year_return': None, 'investment_thesis

## Section 11 — Agent: Portfolio Constructor

The construction agent (OpenAI Agents SDK, `gpt-5-mini`). Unlike the researchers it does **no**
web/API research — it consumes the saved research and the mandate:

**Inputs:** `investment_mandate.json`, `fixed_income_candidates.json`, `etf_candidates.json`,
`equity_candidates.json`.

**Outputs — three portfolios:** **A — Conservative**, **B — Balanced**, **C — Yield Focused**.
For each: selected instruments with per-instrument weights, asset / currency / sector allocation,
expected portfolio yield, and a mandate-compliance check.

> **Reliability note:** the agent *selects* instruments, assigns weights, and buckets each holding
> (Fixed Income / Equity / Liquidity). The allocation breakdowns, expected yield, and the numeric
> compliance checks are then **recomputed deterministically in Python** from those holdings — so the
> saved numbers are always internally consistent rather than trusting LLM mental math.

Saved to `outputs/portfolios/{conservative,balanced,yield}_portfolio.json`.

In [26]:
class PortfolioHolding(BaseModel):
    """A single position within a portfolio."""
    name: str
    ticker: Optional[str] = None
    instrument_type: str          # Corporate Bond | Government Bond | Bond ETF | Money Market ETF | Equity | Equity ETF
    asset_bucket: str             # Fixed Income | Equity | Liquidity  (mandate-level bucket)
    sector: Optional[str] = None
    region: Optional[str] = None  # EU | USA | LATAM
    currency: Optional[str] = None
    weight_pct: float             # weight within this portfolio (0-100)
    expected_yield_pct: Optional[float] = None  # numeric yield, e.g. 6.59
    rationale: Optional[str] = None


class MandateComplianceCheck(BaseModel):
    fixed_income_pct: float
    equity_pct: float
    liquidity_pct: float
    fixed_income_band_ok: bool
    equity_band_ok: bool
    expected_yield_pct: float
    meets_target_yield: bool
    currencies_ok: bool
    regions_ok: bool
    compliant: bool
    notes: Optional[str] = None


class Portfolio(BaseModel):
    name: str                     # e.g. "Portfolio A: Conservative"
    strategy: str                 # Conservative | Balanced | Yield Focused
    objective: Optional[str] = None
    holdings: list[PortfolioHolding]
    asset_allocation: dict[str, float] = Field(default_factory=dict)
    currency_allocation: dict[str, float] = Field(default_factory=dict)
    sector_allocation: dict[str, float] = Field(default_factory=dict)
    expected_portfolio_yield: float = 0.0
    mandate_compliance: Optional[MandateComplianceCheck] = None


class PortfolioSet(BaseModel):
    """Structured output: the three portfolios the agent must produce."""
    conservative: Portfolio
    balanced: Portfolio
    yield_focused: Portfolio

In [27]:
import re

PORTFOLIO_INSTRUCTIONS = """\
You are the **Portfolio Constructor** for an institutional investment desk. You are given the
investment mandate and three pools of pre-researched candidates (fixed income, ETFs, equities).

Build THREE distinct portfolios from the supplied candidates ONLY (do not invent instruments):
  - **Portfolio A: Conservative** — capital preservation. Tilt to government / investment-grade
    bonds, money-market/liquidity, large defensive equities. Fixed income near the TOP of its band,
    equity near the BOTTOM, a healthy liquidity sleeve. Yield may fall below the 7% target.
  - **Portfolio B: Balanced** — balance income and growth. Diversified mix aiming near the 7% net
    target while respecting the bands.
  - **Portfolio C: Yield Focused** — maximize income within the mandate. Tilt to high-yield bonds /
    high-yield bond ETFs and higher-yielding equities; push expected yield toward or above 7% net.

Rules for EVERY portfolio:
  - Respect the mandate allocation bands: Fixed Income and Equity must fall within their stated
    ranges; the remainder is the Liquidity sleeve (money-market / cash-like ETFs).
  - Set each holding's `asset_bucket` to exactly one of: "Fixed Income", "Equity", "Liquidity".
    (Bonds & bond ETFs -> Fixed Income; money-market ETFs -> Liquidity; equities & equity ETFs -> Equity.)
  - Set `region` to one of "EU", "USA", "LATAM" (map the candidate's country/region accordingly).
  - Only use eligible currencies (EUR, USD, RSD).
  - `weight_pct` across all holdings in a portfolio should sum to ~100.
  - Provide a numeric `expected_yield_pct` per holding, taken from the candidate's yield data
    (bond YTM/coupon, ETF SEC yield, or equity dividend yield). Use 0 if genuinely none.
  - Prefer higher-`score` candidates. Aim for 8-15 holdings per portfolio with sensible diversification.

You do not need to compute portfolio-level allocation totals precisely — they will be recomputed
downstream. Focus on a sound selection, correct buckets, and reasonable weights.
"""

portfolio_constructor = Agent(
    name="Portfolio Constructor",
    model="gpt-5-mini",
    instructions=PORTFOLIO_INSTRUCTIONS,
    output_type=AgentOutputSchema(PortfolioSet, strict_json_schema=False),
)


# ---- Deterministic recompute helpers (trustworthy numbers, not LLM math) ----
def _parse_band(text):
    nums = re.findall(r"\d+(?:\.\d+)?", text or "")
    if len(nums) >= 2:
        return float(nums[0]), float(nums[1])
    if len(nums) == 1:
        return float(nums[0]), float(nums[0])
    return None, None


def _recompute_portfolio(portfolio: dict, mandate: dict) -> dict:
    """Normalize weights and recompute allocations / yield / compliance from holdings."""
    holdings = portfolio.get("holdings", [])
    total_w = sum(h.get("weight_pct") or 0 for h in holdings) or 1.0
    for h in holdings:
        h["weight_pct"] = round((h.get("weight_pct") or 0) * 100.0 / total_w, 2)

    def agg(key):
        d = {}
        for h in holdings:
            k = h.get(key) or "Unknown"
            d[k] = round(d.get(k, 0.0) + h["weight_pct"], 2)
        return d

    asset_alloc = agg("asset_bucket")
    currency_alloc = agg("currency")
    sector_alloc = agg("sector")
    exp_yield = round(
        sum((h.get("expected_yield_pct") or 0.0) * h["weight_pct"] / 100.0 for h in holdings), 2
    )

    fi = asset_alloc.get("Fixed Income", 0.0)
    eq = asset_alloc.get("Equity", 0.0)
    liq = asset_alloc.get("Liquidity", 0.0)
    fi_lo, fi_hi = _parse_band(mandate.get("fixed_income_allocation"))
    eq_lo, eq_hi = _parse_band(mandate.get("equity_allocation"))
    ty_lo, _ = _parse_band(mandate.get("target_yield"))

    fi_ok = (fi_lo <= fi <= fi_hi) if fi_lo is not None else True
    eq_ok = (eq_lo <= eq <= eq_hi) if eq_lo is not None else True
    cur_ok = all(h.get("currency") in mandate.get("currencies", [])
                 for h in holdings if h.get("currency"))
    reg_ok = all(h.get("region") in mandate.get("regions", [])
                 for h in holdings if h.get("region"))
    meets_yield = exp_yield >= (ty_lo or 0)

    notes = []
    if not fi_ok:
        notes.append(f"Fixed income {fi}% outside band {mandate.get('fixed_income_allocation')}.")
    if not eq_ok:
        notes.append(f"Equity {eq}% outside band {mandate.get('equity_allocation')}.")
    if not cur_ok:
        notes.append("Contains non-eligible currency.")
    if not reg_ok:
        notes.append("Contains non-eligible region.")
    if not meets_yield:
        notes.append(f"Expected yield {exp_yield}% below target {mandate.get('target_yield')}.")

    portfolio["asset_allocation"] = asset_alloc
    portfolio["currency_allocation"] = currency_alloc
    portfolio["sector_allocation"] = sector_alloc
    portfolio["expected_portfolio_yield"] = exp_yield
    portfolio["mandate_compliance"] = {
        "fixed_income_pct": round(fi, 2),
        "equity_pct": round(eq, 2),
        "liquidity_pct": round(liq, 2),
        "fixed_income_band_ok": fi_ok,
        "equity_band_ok": eq_ok,
        "expected_yield_pct": exp_yield,
        "meets_target_yield": meets_yield,
        "currencies_ok": cur_ok,
        "regions_ok": reg_ok,
        # Band + eligibility define hard compliance; hitting the yield target is an objective,
        # not a constraint (the Conservative book is expected to fall short).
        "compliant": bool(fi_ok and eq_ok and cur_ok and reg_ok),
        "notes": " ".join(notes) if notes else "Within mandate bands and eligibility rules.",
    }
    return portfolio


def _compact(candidates, fields):
    return [{k: c.get(k) for k in fields} for c in candidates]


PORTFOLIO_FILES = {
    "conservative": OUTPUT_DIRS["portfolios"] / "conservative_portfolio.json",
    "balanced": OUTPUT_DIRS["portfolios"] / "balanced_portfolio.json",
    "yield_focused": OUTPUT_DIRS["portfolios"] / "yield_portfolio.json",
}


async def run_portfolio_construction() -> dict:
    """Construct the three portfolios from the mandate + saved candidate pools.

    Returns a dict {conservative, balanced, yield_focused} of the (recomputed) portfolios and
    writes each to outputs/portfolios/.
    """
    mandate = load_investment_mandate()
    research = OUTPUT_DIRS["research"]
    fi = json.load(open(research / "fixed_income_candidates.json", encoding="utf-8"))
    etf = json.load(open(research / "etf_candidates.json", encoding="utf-8"))
    eq = json.load(open(research / "equity_candidates.json", encoding="utf-8"))

    fi_c = _compact(fi, ["name", "ticker", "type", "region", "currency", "rating",
                         "coupon", "ytm", "sec_yield", "score", "contributes_to_target_yield"])
    etf_c = _compact(etf, ["name", "ticker", "etf_category", "region", "currency",
                           "sec_yield", "expense_ratio", "score"])
    eq_c = _compact(eq, ["company_name", "ticker", "sector", "country",
                         "dividend_yield", "score"])

    prompt = (
        "Construct three portfolios (Conservative, Balanced, Yield Focused) from these candidates.\n\n"
        f"INVESTMENT MANDATE:\n{json.dumps(mandate, indent=2)}\n\n"
        f"FIXED INCOME CANDIDATES:\n{json.dumps(fi_c, ensure_ascii=False)}\n\n"
        f"ETF CANDIDATES:\n{json.dumps(etf_c, ensure_ascii=False)}\n\n"
        f"EQUITY CANDIDATES:\n{json.dumps(eq_c, ensure_ascii=False)}\n"
    )

    result = await Runner.run(portfolio_constructor, prompt, max_turns=6)
    pset = result.final_output.model_dump()

    out = {}
    for key in ("conservative", "balanced", "yield_focused"):
        portfolio = _recompute_portfolio(pset[key], mandate)
        with open(PORTFOLIO_FILES[key], "w", encoding="utf-8") as f:
            json.dump(portfolio, f, indent=2, ensure_ascii=False)
        out[key] = portfolio

    log_activity("PortfolioConstructor", "run_portfolio_construction", len(out))
    print("Saved 3 portfolios to outputs/portfolios/:")
    for key, path in PORTFOLIO_FILES.items():
        print(f"  - {out[key]['name']:28s} -> {path.name}")
    return out

In [28]:
portfolios = await run_portfolio_construction()

for key in ("conservative", "balanced", "yield_focused"):
    p = portfolios[key]
    mc = p["mandate_compliance"]
    print(f"\n=== {p['name']} ===")
    print(f"  holdings: {len(p['holdings'])} | expected yield: {p['expected_portfolio_yield']}% "
          f"| compliant: {mc['compliant']}")
    print(f"  asset allocation: {p['asset_allocation']}")
    print(f"  currency allocation: {p['currency_allocation']}")
    print(f"  compliance notes: {mc['notes']}")

{"timestamp": "2026-06-12T16:00:30", "agent": "PortfolioConstructor", "query": "run_portfolio_construction", "result_count": 3}


Saved 3 portfolios to outputs/portfolios/:
  - Conservative                 -> conservative_portfolio.json
  - Balanced                     -> balanced_portfolio.json
  - Yield Focused                -> yield_portfolio.json

=== Conservative ===
  holdings: 12 | expected yield: 4.1% | compliant: True
  asset allocation: {'Fixed Income': 65.0, 'Equity': 15.0, 'Liquidity': 20.0}
  currency allocation: {'USD': 79.0, 'RSD': 6.0, 'EUR': 15.0}
  compliance notes: Expected yield 4.1% below target 7% net.

=== Balanced ===
  holdings: 13 | expected yield: 5.38% | compliant: True
  asset allocation: {'Fixed Income': 62.26, 'Equity': 18.86, 'Liquidity': 18.86}
  currency allocation: {'USD': 92.44, 'EUR': 7.54}
  compliance notes: Expected yield 5.38% below target 7% net.

=== Yield Focused ===
  holdings: 11 | expected yield: 5.35% | compliant: True
  asset allocation: {'Fixed Income': 50.0, 'Equity': 25.0, 'Liquidity': 25.0}
  currency allocation: {'USD': 81.0, 'EUR': 19.0}
  compliance notes: 

## Section 12 — Agent: Investment Committee Reviewer

A critical / adversarial review agent (OpenAI Agents SDK, `gpt-5-mini`) that acts as an
institutional investment committee. It reads the mandate and all three portfolios and produces a
**client-presentation recommendation** as Markdown.

It is explicitly instructed to **challenge every yield assumption** — treating coupon, dividend
yield and total return as distinct concepts, never assuming coupon = YTM, and flagging holdings
whose yield estimates are unreliable or based on incomplete data. To support that critique it is
given an honest **data-provenance note** describing how each `expected_yield_pct` was derived
upstream.

**Per portfolio:** strengths, weaknesses, key risks, mandate-fit score (1–100). **Then** a
*Committee Conclusion* with the recommended portfolio, confidence level, a realistic yield range,
and required changes before client presentation. Saved to
`outputs/reports/investment_committee_review.md`.

In [29]:
COMMITTEE_INSTRUCTIONS = """\
You are the **Investment Committee** of an institutional asset manager. You are reviewing three
proposed portfolios (Conservative, Balanced, Yield Focused) against the client mandate and must
prepare a recommendation for client presentation. Be rigorous, skeptical and specific — your job
is to protect the client, not to flatter the portfolios.

ANALYTICAL DISCIPLINE (mandatory):
  - **Challenge every yield assumption.** The portfolios report an `expected_portfolio_yield` that
    is just a weighted average of per-holding `expected_yield_pct`. Do not take it at face value.
  - Treat **coupon**, **dividend yield**, and **total return** as SEPARATE concepts. A bond's coupon
    is NOT its yield-to-maturity; an equity's dividend yield is NOT its total return; an ETF's SEC
    yield is backward/forward-looking and net of fees in different ways.
  - Never assume coupon == YTM. Where a bond's figure is a coupon, say its true forward yield is
    uncertain.
  - **Distinguish gross vs net.** The mandate target is 7% NET. Subtract realistic drags: fund
    expense ratios, FX hedging cost (RSD/EUR/USD mismatch), expected credit losses on high-yield,
    transaction costs, and taxes/withholding. State how these erode the headline number.
  - Identify specific holdings whose yield estimates are unreliable, unsupported, or based on
    incomplete data, and explain why (use the provenance note below).

Use the provided portfolio data and provenance note. Do not invent instruments or numbers; if a
figure is unknown, say so explicitly rather than guessing.

Produce the review in GitHub-flavored Markdown with EXACTLY this structure:

# Investment Committee Review

(1-2 sentence framing of the review.)

## Portfolio A — Conservative
**Strengths**
- ...
**Weaknesses**
- ...
**Key Risks**
- ...
**Mandate Fit Score:** NN/100

## Portfolio B — Balanced
(same four subsections)

## Portfolio C — Yield Focused
(same four subsections)

## Yield Reality Check
- Challenge the headline expected yields; restate a realistic NET yield range and the drags applied.
- List specific holdings with unreliable / unsupported / incomplete yield estimates.

## Spirit-of-Mandate Assessment
- Note any portfolio that violates the SPIRIT of the mandate even if it passes technical band checks
  (e.g. concentration, hidden credit/FX risk, illiquidity, reliance on a single yield source).

## Committee Conclusion
- **Recommended Portfolio:** ...
- **Confidence Level:** Low / Medium / High
- **Expected Realistic Yield Range:** e.g. "4.5%-5.5% net"
- **Required Changes Before Client Presentation:**
  - ...

Also explicitly answer, within the relevant sections above: which portfolio best satisfies the
mandate, whether the 7% target looks realistic under current conditions, the biggest risks, which
assumptions may be wrong/optimistic, and which portfolio to present to the client.
"""

investment_committee = Agent(
    name="Investment Committee Reviewer",
    model="gpt-5-mini",
    instructions=COMMITTEE_INSTRUCTIONS,
    # No output_type: the committee returns a free-form Markdown report.
)

COMMITTEE_OUTPUT_PATH = OUTPUT_DIRS["reports"] / "investment_committee_review.md"

# Honest description of how upstream `expected_yield_pct` values were derived, so the committee
# can challenge them concretely rather than assume they are clean forward yields.
YIELD_PROVENANCE_NOTE = """\
DATA PROVENANCE / KNOWN LIMITATIONS of the per-holding `expected_yield_pct` figures:
- Equities: figure is the TRAILING DIVIDEND YIELD only (excludes price appreciation / total return);
  P/E, analyst target price and 1-year return were NOT collected for equities.
- Bonds: figure may be a stated COUPON or an approximate YTM and the two were not always
  distinguished upstream; treat any single bond yield as uncertain unless clearly a market YTM.
- ETFs: figure is the quoted SEC / distribution yield; average credit rating for bond ETFs is
  largely MISSING, so credit quality of the FI sleeve is not fully verified.
- Money-market / liquidity holdings often carry an `expected_yield_pct` of 0 in the data, which
  understates their real (low) yield but should NOT be inflated.
- All figures are point-in-time, web-sourced and only lightly validated; none are net of fees,
  FX hedging, taxes, or expected credit losses.
"""


async def run_investment_committee_review() -> str:
    """Run the Investment Committee review over the mandate + three portfolios.

    Returns the Markdown review and writes it to
    outputs/reports/investment_committee_review.md.
    """
    mandate = load_investment_mandate()
    portfolios_dir = OUTPUT_DIRS["portfolios"]
    conservative = json.load(open(portfolios_dir / "conservative_portfolio.json", encoding="utf-8"))
    balanced = json.load(open(portfolios_dir / "balanced_portfolio.json", encoding="utf-8"))
    yield_focused = json.load(open(portfolios_dir / "yield_portfolio.json", encoding="utf-8"))

    prompt = (
        "Review these three portfolios against the client mandate and produce the committee report.\n\n"
        f"INVESTMENT MANDATE:\n{json.dumps(mandate, indent=2)}\n\n"
        f"{YIELD_PROVENANCE_NOTE}\n"
        f"PORTFOLIO A - CONSERVATIVE:\n{json.dumps(conservative, ensure_ascii=False)}\n\n"
        f"PORTFOLIO B - BALANCED:\n{json.dumps(balanced, ensure_ascii=False)}\n\n"
        f"PORTFOLIO C - YIELD FOCUSED:\n{json.dumps(yield_focused, ensure_ascii=False)}\n"
    )

    result = await Runner.run(investment_committee, prompt, max_turns=4)
    review_md = result.final_output

    with open(COMMITTEE_OUTPUT_PATH, "w", encoding="utf-8") as f:
        f.write(review_md)

    log_activity("InvestmentCommitteeReviewer", "run_investment_committee_review", 3)
    print(f"Saved committee review to {COMMITTEE_OUTPUT_PATH}\n")
    return review_md

In [30]:
review = await run_investment_committee_review()

print(review)

{"timestamp": "2026-06-12T16:01:37", "agent": "InvestmentCommitteeReviewer", "query": "run_investment_committee_review", "result_count": 3}


Saved committee review to /Users/ogurianova/Desktop/ANTON TASK1/outputs/reports/investment_committee_review.md

# Investment Committee Review

We reviewed the three proposed portfolios versus the client mandate (RSD 100M, 7% net target, 50–65% fixed income, 15–25% equity, min bond rating B-, 3+ year horizon, liquid instruments). All three are technically within the allocation bands but none demonstrate a credible path to 7% net without material additional risk or untested assumptions.

## Portfolio A — Conservative
**Strengths**
- High-quality bias and capital-preservation focus; fixed income at 65% meets mandate band.
- Liquid instruments; currency and region constraints respected.
- Good sector alignment for equities (Healthcare, Chemicals).

**Weaknesses**
- Headline expected yield 4.1% far below 7% net; cannot meet mandate income target without changing risk profile.
- Heavy allocation to USD cash/short-term treasuries (20% liquidity) drags yield.
- Several bond yields reported may

## Section 13 — Agent: Investment Proposal Writer

A senior-advisor agent (OpenAI Agents SDK, `gpt-5-mini`) that reads the mandate, all three
portfolios, and the Investment Committee review and produces a **12-slide, client-ready investment
proposal** in two formats:

* `outputs/reports/investment_proposal.md` — full Markdown text (one `##` section per slide)
* `outputs/reports/investment_proposal.json` — structured slide-by-slide JSON

The agent uses the committee's recommendation to select the portfolio, writes in a professional
institutional tone, clearly separates income yield from total return, and flags every material
assumption or limitation. It never exaggerates expected returns.

**Slide inventory:** Executive Summary · Market Environment · Portfolio Recommendation ·
Asset Allocation · Currency Allocation · Sector Allocation · Top Holdings · Risk Assessment ·
Portfolio Comparison · Committee Recommendation · Expected Outcomes · Conclusion & Next Steps.

In [31]:
PROPOSAL_INSTRUCTIONS = """\
You are a **Senior Investment Advisor** at an institutional asset management firm preparing a
formal, client-facing investment proposal presentation.

TONE AND DISCIPLINE (mandatory):
  - Use professional institutional language throughout. No marketing hyperbole.
  - **Never exaggerate expected returns.** Present realistic, net-of-cost yield ranges.
  - **Distinguish income yield from total return** explicitly wherever both are relevant.
  - Label every material assumption and limitation clearly.
  - Numbers must be presentation-friendly (e.g. '5.2%', 'RSD 10 M', '60/30/10').
  - Follow the Investment Committee conclusions — select the committee-recommended portfolio.

OUTPUT REQUIREMENTS:
You must produce TWO artefacts inside a single JSON response with keys 'markdown' and 'slides'.

1. 'markdown': a single Markdown string containing the full proposal, one ## section per slide.

2. 'slides': a JSON array of exactly 12 slide objects. Each object has:
   { "slide_number": int, "title": str, "content": { ... slide-specific keys ... } }

Slide schemas (use EXACTLY these keys):

Slide 1 — Executive Summary
  content: { "client_objectives": str, "investment_horizon": str, "key_recommendation": str }

Slide 2 — Market Environment
  content: { "current_fixed_income_environment": str, "yield_opportunities": str,
             "key_assumptions": [str, ...] }

Slide 3 — Portfolio Recommendation
  content: { "recommended_portfolio": str, "expected_yield": str, "risk_profile": str,
             "key_benefits": [str, ...] }

Slide 4 — Asset Allocation
  content: { "fixed_income_pct": float, "equity_pct": float, "liquidity_pct": float,
             "pie_chart_data": [{"label": str, "value": float}, ...] }

Slide 5 — Currency Allocation
  content: { "usd_pct": float, "eur_pct": float, "rsd_pct": float,
             "pie_chart_data": [{"label": str, "value": float}, ...] }

Slide 6 — Sector Allocation
  content: { "sector_breakdown": [{"sector": str, "pct": float}, ...],
             "key_sector_exposures": str }

Slide 7 — Top Holdings
  content: { "holdings": [{"instrument": str, "weight_pct": float,
              "expected_yield": str, "investment_rationale": str}, ...] }

Slide 8 — Risk Assessment
  content: { "credit_risk": str, "interest_rate_risk": str,
             "currency_risk": str, "liquidity_risk": str }

Slide 9 — Portfolio Comparison
  content: { "comparison": [
    {"portfolio": "Conservative", "expected_yield": str, "risk_level": str, "mandate_fit_score": int},
    {"portfolio": "Balanced",     "expected_yield": str, "risk_level": str, "mandate_fit_score": int},
    {"portfolio": "Yield Focused","expected_yield": str, "risk_level": str, "mandate_fit_score": int}
  ] }

Slide 10 — Committee Recommendation
  content: { "why_selected": str, "confidence_level": str, "key_assumptions": [str, ...] }

Slide 11 — Expected Outcomes
  content: { "realistic_yield_range": str, "upside_scenario": str, "downside_scenario": str }

Slide 12 — Conclusion & Next Steps
  content: { "conclusion": str, "next_steps": [str, ...] }

Return ONLY valid JSON. Do not wrap in markdown fences.
"""

from pydantic import field_validator


class ProposalOutput(BaseModel):
    """Structured output from the Investment Proposal Writer."""
    markdown: str
    slides: list
    executive_summary: str = ""

    @field_validator("executive_summary", mode="before")
    @classmethod
    def _default_executive_summary(cls, v, info):
        # Convenience accessor: return the Slide 1 content block as a formatted string.
        if v:
            return v
        slides = info.data.get("slides", [])
        if slides:
            s1 = slides[0]
            c = s1.get("content", {})
            return (
                f"Objectives: {c.get('client_objectives', '')}\n"
                f"Horizon: {c.get('investment_horizon', '')}\n"
                f"Recommendation: {c.get('key_recommendation', '')}"
            )
        return ""


investment_proposal_writer = Agent(
    name="Investment Proposal Writer",
    model="gpt-5-mini",
    instructions=PROPOSAL_INSTRUCTIONS,
)

PROPOSAL_MD_PATH   = OUTPUT_DIRS["reports"] / "investment_proposal.md"
PROPOSAL_JSON_PATH = OUTPUT_DIRS["reports"] / "investment_proposal.json"


async def run_investment_proposal_writer() -> ProposalOutput:
    """Run the Investment Proposal Writer.

    Reads:
      - outputs/context/investment_mandate.json
      - outputs/portfolios/conservative_portfolio.json
      - outputs/portfolios/balanced_portfolio.json
      - outputs/portfolios/yield_portfolio.json
      - outputs/reports/investment_committee_review.md

    Writes:
      - outputs/reports/investment_proposal.md
      - outputs/reports/investment_proposal.json

    Returns a ProposalOutput with .markdown, .slides, and .executive_summary.
    """
    mandate = load_investment_mandate()
    portfolios_dir = OUTPUT_DIRS["portfolios"]
    conservative  = json.load(open(portfolios_dir / "conservative_portfolio.json", encoding="utf-8"))
    balanced      = json.load(open(portfolios_dir / "balanced_portfolio.json",      encoding="utf-8"))
    yield_focused = json.load(open(portfolios_dir / "yield_portfolio.json",          encoding="utf-8"))
    committee_review = COMMITTEE_OUTPUT_PATH.read_text(encoding="utf-8")

    prompt = (
        "Prepare a 12-slide client investment proposal based on the data below.\n\n"
        f"INVESTMENT MANDATE:\n{json.dumps(mandate, indent=2)}\n\n"
        f"PORTFOLIO — CONSERVATIVE:\n{json.dumps(conservative, ensure_ascii=False)}\n\n"
        f"PORTFOLIO — BALANCED:\n{json.dumps(balanced, ensure_ascii=False)}\n\n"
        f"PORTFOLIO — YIELD FOCUSED:\n{json.dumps(yield_focused, ensure_ascii=False)}\n\n"
        f"INVESTMENT COMMITTEE REVIEW:\n{committee_review}\n"
    )

    result = await Runner.run(investment_proposal_writer, prompt, max_turns=4)
    raw = result.final_output

    # Strip optional markdown fences the model may add despite instructions.
    raw = re.sub(r"^```(?:json)?\s*", "", raw.strip(), flags=re.MULTILINE)
    raw = re.sub(r"```\s*$", "", raw.strip(), flags=re.MULTILINE)

    parsed = json.loads(raw)
    proposal = ProposalOutput(**parsed)

    # Ensure executive_summary is populated.
    if not proposal.executive_summary and proposal.slides:
        s1 = proposal.slides[0]
        c  = s1.get("content", {})
        proposal.executive_summary = (
            f"Objectives: {c.get('client_objectives', '')}\n"
            f"Horizon: {c.get('investment_horizon', '')}\n"
            f"Recommendation: {c.get('key_recommendation', '')}"
        )

    # Persist outputs.
    with open(PROPOSAL_MD_PATH, "w", encoding="utf-8") as f:
        f.write(proposal.markdown)
    with open(PROPOSAL_JSON_PATH, "w", encoding="utf-8") as f:
        json.dump({"slides": proposal.slides}, f, ensure_ascii=False, indent=2)

    log_activity("InvestmentProposalWriter", "run_investment_proposal_writer", 12)
    print(f"Saved investment proposal to:\n  {PROPOSAL_MD_PATH}\n  {PROPOSAL_JSON_PATH}\n")
    return proposal


In [32]:
proposal = await run_investment_proposal_writer()

print(proposal.executive_summary)


{"timestamp": "2026-06-12T16:02:58", "agent": "InvestmentProposalWriter", "query": "run_investment_proposal_writer", "result_count": 12}


Saved investment proposal to:
  /Users/ogurianova/Desktop/ANTON TASK1/outputs/reports/investment_proposal.md
  /Users/ogurianova/Desktop/ANTON TASK1/outputs/reports/investment_proposal.json

Objectives: Preserve capital in liquid instruments; target income; RSD 100 M principal; target 7% net (client stated).
Horizon: 3+ years; liquid-only mandate; fixed income 50–65%, equity 15–25%.
Recommendation: Present Investment Committee-recommended portfolio: Balanced (Portfolio B). Subject to verification steps and re-stated net-yield scenarios. Realistic net yield range estimated at 3.8%–4.9% (illustrative).


## Section 14 — Presentation Generator

A pure-Python presentation builder that reads `investment_proposal.json`,
`investment_committee_review.md`, and `investment_mandate.json` and produces a polished 13-slide
PowerPoint deck with embedded charts and tables.

**Charts generated:**
- Asset Allocation pie chart (Slide 5)
- Currency Allocation pie chart (Slide 6)
- Sector Allocation pie chart (Slide 7)
- Portfolio Comparison bar chart (Slide 9)

**Outputs:**
- `outputs/presentations/investment_proposal.pptx`
- `outputs/presentations/investment_proposal.pdf`  *(via LibreOffice if available)*
- `outputs/presentations/presentation_metadata.json`

**Styling:** white background, dark-blue (`1B3A6B`) headings, Calibri/Calibri Light typography,
clean bordered tables, executive institutional look.


In [33]:
%pip install -q python-pptx matplotlib

import io
import json
import datetime as dt
from pathlib import Path
from typing import Any

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.util import Inches, Pt, Emu
from pptx.enum.text import PP_ALIGN
from pptx.dml.color import RGBColor
from pptx.util import Inches, Pt


# ── Design tokens ─────────────────────────────────────────────────────────────
C_NAVY      = RGBColor(0x1B, 0x3A, 0x6B)   # headings / accents
C_WHITE     = RGBColor(0xFF, 0xFF, 0xFF)
C_OFFWHITE  = RGBColor(0xF5, 0xF7, 0xFA)
C_LIGHTGRAY = RGBColor(0xD6, 0xDC, 0xE4)
C_DARKTEXT  = RGBColor(0x1A, 0x1A, 0x2E)
C_SUBTEXT   = RGBColor(0x4A, 0x5A, 0x6A)
C_ACCENT    = RGBColor(0x2E, 0x86, 0xAB)   # teal accent

CHART_PALETTE = ["1B3A6B", "2E86AB", "A8DADC", "457B9D", "E63946", "F4A261", "2A9D8F"]

PRESENTATIONS_DIR = BASE_DIR / "outputs" / "presentations"
PRESENTATIONS_DIR.mkdir(parents=True, exist_ok=True)

PPTX_PATH     = PRESENTATIONS_DIR / "investment_proposal.pptx"
PDF_PATH      = PRESENTATIONS_DIR / "investment_proposal.pdf"
METADATA_PATH = PRESENTATIONS_DIR / "presentation_metadata.json"


# ── Helpers ───────────────────────────────────────────────────────────────────

def _rgb_hex(hex_str: str) -> RGBColor:
    h = hex_str.lstrip("#")
    return RGBColor(int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16))


def _set_font(run, size_pt: int, bold: bool = False, color: RGBColor = None, italic: bool = False):
    run.font.size = Pt(size_pt)
    run.font.bold = bold
    run.font.italic = italic
    run.font.name = "Calibri"
    if color:
        run.font.color.rgb = color


def _add_slide(prs: Presentation) -> Any:
    """Add a blank slide (layout 6 = blank)."""
    blank_layout = prs.slide_layouts[6]
    return prs.slides.add_slide(blank_layout)


def _header_bar(slide, title_text: str, prs: Presentation):
    """Navy top bar + white title text."""
    from pptx.util import Inches, Pt, Emu
    W = prs.slide_width
    # Navy bar
    bar = slide.shapes.add_shape(
        1,  # MSO_SHAPE_TYPE.RECTANGLE
        0, 0, W, Inches(0.72)
    )
    bar.fill.solid()
    bar.fill.fore_color.rgb = C_NAVY
    bar.line.fill.background()

    # Title text
    txb = slide.shapes.add_textbox(Inches(0.35), Inches(0.10), W - Inches(0.70), Inches(0.52))
    tf = txb.text_frame
    tf.word_wrap = False
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.LEFT
    run = p.add_run()
    run.text = title_text
    _set_font(run, 22, bold=True, color=C_WHITE)


def _footer(slide, page_num: int, total: int, prs: Presentation):
    """Subtle footer with page number."""
    W = prs.slide_width
    H = prs.slide_height
    txb = slide.shapes.add_textbox(Inches(0.3), H - Inches(0.32), W - Inches(0.6), Inches(0.28))
    tf = txb.text_frame
    p = tf.paragraphs[0]
    p.alignment = PP_ALIGN.RIGHT
    run = p.add_run()
    run.text = f"{page_num} / {total}  |  CONFIDENTIAL — FOR CLIENT USE ONLY"
    _set_font(run, 8, color=C_SUBTEXT)


def _body_box(slide, text_lines: list[str], x, y, w, h,
              font_size: int = 13, color: RGBColor = None):
    """Multi-line text box with bullet-style lines."""
    color = color or C_DARKTEXT
    txb = slide.shapes.add_textbox(x, y, w, h)
    tf = txb.text_frame
    tf.word_wrap = True
    for i, line in enumerate(text_lines):
        p = tf.paragraphs[0] if i == 0 else tf.add_paragraph()
        p.space_after = Pt(4)
        run = p.add_run()
        run.text = line
        _set_font(run, font_size, color=color)


def _section_label(slide, label: str, x, y, w=Inches(3.5), h=Inches(0.32)):
    """Small navy section label."""
    txb = slide.shapes.add_textbox(x, y, w, h)
    tf = txb.text_frame
    p = tf.paragraphs[0]
    run = p.add_run()
    run.text = label.upper()
    _set_font(run, 9, bold=True, color=C_ACCENT)


def _kv_block(slide, pairs: list[tuple[str, str]], x, y, w, row_h=Inches(0.34), font_size=12):
    """Key / value label pairs stacked vertically."""
    for i, (key, val) in enumerate(pairs):
        yy = y + i * row_h
        # Key label
        kb = slide.shapes.add_textbox(x, yy, w * 0.42, row_h)
        ktf = kb.text_frame
        kp = ktf.paragraphs[0]
        kr = kp.add_run()
        kr.text = key
        _set_font(kr, font_size, bold=True, color=C_NAVY)

        # Value
        vb = slide.shapes.add_textbox(x + w * 0.44, yy, w * 0.56, row_h)
        vtf = vb.text_frame
        vtf.word_wrap = True
        vp = vtf.paragraphs[0]
        vr = vp.add_run()
        vr.text = str(val)
        _set_font(vr, font_size, color=C_DARKTEXT)


def _add_table(slide, headers: list[str], rows: list[list[str]],
               x, y, w, col_widths: list[float] = None):
    """Styled table with navy header row."""
    n_cols = len(headers)
    n_rows = len(rows) + 1
    row_h = Inches(0.34)

    tbl = slide.shapes.add_table(n_rows, n_cols, x, y, w, row_h * n_rows).table

    # Column widths
    if col_widths:
        for ci, cw in enumerate(col_widths):
            tbl.columns[ci].width = Inches(cw)

    # Header row
    for ci, hdr in enumerate(headers):
        cell = tbl.cell(0, ci)
        cell.fill.solid()
        cell.fill.fore_color.rgb = C_NAVY
        p = cell.text_frame.paragraphs[0]
        p.alignment = PP_ALIGN.CENTER
        run = p.add_run()
        run.text = hdr
        _set_font(run, 11, bold=True, color=C_WHITE)

    # Data rows
    for ri, row in enumerate(rows):
        bg = C_OFFWHITE if ri % 2 == 0 else C_WHITE
        for ci, val in enumerate(row):
            cell = tbl.cell(ri + 1, ci)
            cell.fill.solid()
            cell.fill.fore_color.rgb = bg
            p = cell.text_frame.paragraphs[0]
            p.alignment = PP_ALIGN.CENTER
            run = p.add_run()
            run.text = str(val)
            _set_font(run, 11, color=C_DARKTEXT)

    return tbl


def _pie_to_image(labels: list[str], values: list[float], title: str) -> bytes:
    """Return a PNG pie chart as bytes."""
    colors = [f"#{c}" for c in CHART_PALETTE[:len(labels)]]
    fig, ax = plt.subplots(figsize=(5, 4), facecolor="white")
    wedges, texts, autotexts = ax.pie(
        values, labels=labels, colors=colors,
        autopct="%1.1f%%", startangle=90,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
        textprops={"fontsize": 10}
    )
    for at in autotexts:
        at.set_color("white")
        at.set_fontweight("bold")
    ax.set_title(title, fontsize=12, fontweight="bold", color="#1B3A6B", pad=10)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def _bar_to_image(categories: list[str], series: dict[str, list[float]], title: str) -> bytes:
    """Return a PNG grouped bar chart as bytes."""
    import numpy as np
    x = np.arange(len(categories))
    n = len(series)
    width = 0.25
    colors = [f"#{c}" for c in CHART_PALETTE[:n]]

    fig, ax = plt.subplots(figsize=(8, 4), facecolor="white")
    for i, (label, vals) in enumerate(series.items()):
        offset = (i - n / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=label, color=colors[i], edgecolor="white")
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.1,
                    f"{h:.1f}", ha="center", va="bottom", fontsize=8, color="#1A1A2E")

    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=10)
    ax.set_title(title, fontsize=12, fontweight="bold", color="#1B3A6B", pad=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, color="#E2E8F0", linewidth=0.8)
    ax.set_axisbelow(True)
    ax.legend(fontsize=9)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=150, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return buf.read()


def _img_stream(img_bytes: bytes) -> io.BytesIO:
    s = io.BytesIO(img_bytes)
    s.seek(0)
    return s


# ── Slide builders ────────────────────────────────────────────────────────────
TOTAL_SLIDES = 13


def _slide_title(prs, mandate: dict):
    slide = _add_slide(prs)
    W = prs.slide_width
    H = prs.slide_height

    # Full navy background
    bg = slide.shapes.add_shape(1, 0, 0, W, H)
    bg.fill.solid()
    bg.fill.fore_color.rgb = C_NAVY
    bg.line.fill.background()

    # White accent bar
    bar = slide.shapes.add_shape(1, 0, Inches(2.7), Inches(0.08), Inches(2.5))
    bar.fill.solid()
    bar.fill.fore_color.rgb = C_ACCENT
    bar.line.fill.background()

    for text, y, sz, bold in [
        ("INVESTMENT PROPOSAL", Inches(1.6), 34, True),
        (mandate.get("client_name", "Client"), Inches(2.2), 22, False),
        (mandate.get("portfolio_size", ""), Inches(2.85), 16, False),
        (f"Prepared: {dt.date.today().strftime('%B %Y')}", Inches(4.5), 12, False),
        ("CONFIDENTIAL", Inches(4.95), 10, False),
    ]:
        txb = slide.shapes.add_textbox(Inches(0.6), y, W - Inches(1.2), Inches(0.6))
        tf = txb.text_frame
        p = tf.paragraphs[0]
        p.alignment = PP_ALIGN.LEFT
        run = p.add_run()
        run.text = text
        _set_font(run, sz, bold=bold, color=C_WHITE)


def _slide_exec_summary(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Executive Summary", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})
    pairs = [
        ("Client Objectives",   c.get("client_objectives", "—")),
        ("Investment Horizon",  c.get("investment_horizon", "—")),
        ("Key Recommendation",  c.get("key_recommendation", "—")),
    ]
    _kv_block(slide, pairs, Inches(0.4), Inches(0.9), Inches(9.2),
              row_h=Inches(1.3), font_size=13)


def _slide_market_env(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Market Environment", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})

    _section_label(slide, "Current Fixed Income Environment", Inches(0.4), Inches(0.82))
    _body_box(slide, [c.get("current_fixed_income_environment", "—")],
              Inches(0.4), Inches(1.10), Inches(9.2), Inches(1.0), font_size=12)

    _section_label(slide, "Yield Opportunities", Inches(0.4), Inches(2.15))
    _body_box(slide, [c.get("yield_opportunities", "—")],
              Inches(0.4), Inches(2.43), Inches(9.2), Inches(0.8), font_size=12)

    _section_label(slide, "Key Assumptions", Inches(0.4), Inches(3.30))
    assumptions = c.get("key_assumptions", [])
    _body_box(slide, [f"• {a}" for a in assumptions],
              Inches(0.4), Inches(3.58), Inches(9.2), Inches(1.5), font_size=11)


def _slide_recommended_portfolio(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Recommended Portfolio", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})

    # Left card
    card = slide.shapes.add_shape(1, Inches(0.35), Inches(0.85), Inches(4.4), Inches(4.3))
    card.fill.solid()
    card.fill.fore_color.rgb = C_OFFWHITE
    card.line.color.rgb = C_LIGHTGRAY

    pairs = [
        ("Portfolio",       c.get("recommended_portfolio", "—")),
        ("Expected Yield",  c.get("expected_yield", "—")),
        ("Risk Profile",    c.get("risk_profile", "—")),
    ]
    _kv_block(slide, pairs, Inches(0.55), Inches(1.0), Inches(4.0),
              row_h=Inches(0.9), font_size=12)

    # Right: key benefits
    _section_label(slide, "Key Benefits", Inches(5.0), Inches(0.82))
    benefits = c.get("key_benefits", [])
    _body_box(slide, [f"✓  {b}" for b in benefits],
              Inches(5.0), Inches(1.10), Inches(4.7), Inches(4.0), font_size=12)


def _slide_asset_allocation(prs, slide_data: dict, num: int, chart_bytes: bytes):
    slide = _add_slide(prs)
    _header_bar(slide, "Asset Allocation", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})

    # KV on left
    pairs = [
        ("Fixed Income",  f"{c.get('fixed_income_pct', 0):.1f}%"),
        ("Equity",        f"{c.get('equity_pct', 0):.1f}%"),
        ("Liquidity",     f"{c.get('liquidity_pct', 0):.1f}%"),
    ]
    _kv_block(slide, pairs, Inches(0.4), Inches(0.9), Inches(3.8),
              row_h=Inches(0.85), font_size=14)

    # Pie chart on right
    slide.shapes.add_picture(_img_stream(chart_bytes),
                             Inches(4.0), Inches(0.75), Inches(5.7), Inches(4.5))


def _slide_currency_allocation(prs, slide_data: dict, num: int, chart_bytes: bytes):
    slide = _add_slide(prs)
    _header_bar(slide, "Currency Allocation", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})
    pairs = [
        ("USD", f"{c.get('usd_pct', 0):.1f}%"),
        ("EUR", f"{c.get('eur_pct', 0):.1f}%"),
        ("RSD", f"{c.get('rsd_pct', 0):.1f}%"),
    ]
    _kv_block(slide, pairs, Inches(0.4), Inches(0.9), Inches(3.8),
              row_h=Inches(0.85), font_size=14)
    slide.shapes.add_picture(_img_stream(chart_bytes),
                             Inches(4.0), Inches(0.75), Inches(5.7), Inches(4.5))


def _slide_sector_allocation(prs, slide_data: dict, num: int, chart_bytes: bytes):
    slide = _add_slide(prs)
    _header_bar(slide, "Sector Allocation", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})

    # Table on left
    breakdown = c.get("sector_breakdown", [])
    rows = [[s.get("sector", "—"), f"{s.get('pct', 0):.1f}%"] for s in breakdown]
    if rows:
        _add_table(slide, ["Sector", "Weight"],
                   rows, Inches(0.4), Inches(0.85), Inches(4.0),
                   col_widths=[2.8, 1.2])

    # Pie on right
    slide.shapes.add_picture(_img_stream(chart_bytes),
                             Inches(4.6), Inches(0.75), Inches(5.1), Inches(4.5))

    _section_label(slide, "Key Exposures", Inches(0.4), Inches(4.5))
    _body_box(slide, [c.get("key_sector_exposures", "—")],
              Inches(0.4), Inches(4.78), Inches(9.2), Inches(0.6), font_size=11)


def _slide_top_holdings(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Top Holdings", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    holdings = slide_data.get("content", {}).get("holdings", [])
    rows = [
        [
            h.get("instrument", "—"),
            f"{h.get('weight_pct', 0):.1f}%",
            h.get("expected_yield", "—"),
            h.get("investment_rationale", "—"),
        ]
        for h in holdings[:8]
    ]
    _add_table(slide,
               ["Instrument", "Weight", "Exp. Yield", "Rationale"],
               rows,
               Inches(0.3), Inches(0.85), Inches(9.4),
               col_widths=[2.5, 0.85, 0.95, 5.05])


def _slide_portfolio_comparison(prs, slide_data: dict, num: int, chart_bytes: bytes):
    slide = _add_slide(prs)
    _header_bar(slide, "Portfolio Comparison", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    comparison = slide_data.get("content", {}).get("comparison", [])
    rows = [
        [
            c.get("portfolio", "—"),
            c.get("expected_yield", "—"),
            c.get("risk_level", "—"),
            str(c.get("mandate_fit_score", "—")),
        ]
        for c in comparison
    ]
    _add_table(slide,
               ["Portfolio", "Expected Yield", "Risk Level", "Mandate Fit"],
               rows,
               Inches(0.3), Inches(0.85), Inches(9.4),
               col_widths=[2.5, 2.0, 2.4, 2.5])

    slide.shapes.add_picture(_img_stream(chart_bytes),
                             Inches(0.3), Inches(2.35), Inches(9.4), Inches(2.95))


def _slide_risk_assessment(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Risk Assessment", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})
    risks = [
        ("Credit Risk",        c.get("credit_risk", "—")),
        ("Interest Rate Risk", c.get("interest_rate_risk", "—")),
        ("Currency Risk",      c.get("currency_risk", "—")),
        ("Liquidity Risk",     c.get("liquidity_risk", "—")),
    ]

    for i, (label, desc) in enumerate(risks):
        col = i % 2
        row = i // 2
        x = Inches(0.35) + col * Inches(4.85)
        y = Inches(0.85) + row * Inches(2.25)

        card = slide.shapes.add_shape(1, x, y, Inches(4.55), Inches(2.1))
        card.fill.solid()
        card.fill.fore_color.rgb = C_OFFWHITE
        card.line.color.rgb = C_LIGHTGRAY

        lbl = slide.shapes.add_textbox(x + Inches(0.15), y + Inches(0.10),
                                       Inches(4.25), Inches(0.32))
        ltf = lbl.text_frame
        lp = ltf.paragraphs[0]
        lr = lp.add_run()
        lr.text = label
        _set_font(lr, 12, bold=True, color=C_NAVY)

        dtxb = slide.shapes.add_textbox(x + Inches(0.15), y + Inches(0.46),
                                        Inches(4.25), Inches(1.5))
        dtf = dtxb.text_frame
        dtf.word_wrap = True
        dp = dtf.paragraphs[0]
        dr = dp.add_run()
        dr.text = desc
        _set_font(dr, 11, color=C_DARKTEXT)


def _slide_expected_outcomes(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Expected Outcomes", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})
    scenarios = [
        ("Base Case",  "Realistic Yield Range", c.get("realistic_yield_range", "—"), "2E86AB"),
        ("Upside",     "Optimistic Scenario",   c.get("upside_scenario", "—"),       "2A9D8F"),
        ("Downside",   "Stress Scenario",       c.get("downside_scenario", "—"),     "E63946"),
    ]
    for i, (tag, label, text, color_hex) in enumerate(scenarios):
        x = Inches(0.3) + i * Inches(3.2)
        card = slide.shapes.add_shape(1, x, Inches(0.85), Inches(3.05), Inches(4.35))
        card.fill.solid()
        card.fill.fore_color.rgb = C_OFFWHITE
        card.line.color.rgb = C_LIGHTGRAY

        accent = slide.shapes.add_shape(1, x, Inches(0.85), Inches(3.05), Inches(0.06))
        accent.fill.solid()
        accent.fill.fore_color.rgb = _rgb_hex(color_hex)
        accent.line.fill.background()

        for txt, yy, sz, bold, col in [
            (tag,   Inches(0.98), 10, True,  C_ACCENT),
            (label, Inches(1.22), 13, True,  C_NAVY),
            (text,  Inches(1.60), 11, False, C_DARKTEXT),
        ]:
            txb = slide.shapes.add_textbox(x + Inches(0.15), yy, Inches(2.75), Inches(2.6))
            tf = txb.text_frame
            tf.word_wrap = True
            p = tf.paragraphs[0]
            r = p.add_run()
            r.text = txt
            _set_font(r, sz, bold=bold, color=col)


def _slide_committee_recommendation(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Committee Recommendation", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})
    pairs = [
        ("Why Selected",    c.get("why_selected", "—")),
        ("Confidence",      c.get("confidence_level", "—")),
    ]
    _kv_block(slide, pairs, Inches(0.4), Inches(0.9), Inches(9.2),
              row_h=Inches(1.1), font_size=12)

    _section_label(slide, "Key Assumptions", Inches(0.4), Inches(3.20))
    assumptions = c.get("key_assumptions", [])
    _body_box(slide, [f"• {a}" for a in assumptions],
              Inches(0.4), Inches(3.48), Inches(9.2), Inches(1.8), font_size=11)


def _slide_next_steps(prs, slide_data: dict, num: int):
    slide = _add_slide(prs)
    _header_bar(slide, "Conclusion & Next Steps", prs)
    _footer(slide, num, TOTAL_SLIDES, prs)

    c = slide_data.get("content", {})

    _section_label(slide, "Conclusion", Inches(0.4), Inches(0.82))
    _body_box(slide, [c.get("conclusion", "—")],
              Inches(0.4), Inches(1.10), Inches(9.2), Inches(1.6), font_size=12)

    _section_label(slide, "Next Steps", Inches(0.4), Inches(2.80))
    steps = c.get("next_steps", [])
    for i, step in enumerate(steps[:6]):
        x = Inches(0.4) + (i % 2) * Inches(4.75)
        y = Inches(3.10) + (i // 2) * Inches(0.72)
        num_box = slide.shapes.add_shape(1, x, y + Inches(0.04), Inches(0.34), Inches(0.34))
        num_box.fill.solid()
        num_box.fill.fore_color.rgb = C_NAVY
        num_box.line.fill.background()

        ntxb = slide.shapes.add_textbox(x, y, Inches(0.34), Inches(0.38))
        ntf = ntxb.text_frame
        np_ = ntf.paragraphs[0]
        np_.alignment = PP_ALIGN.CENTER
        nr = np_.add_run()
        nr.text = str(i + 1)
        _set_font(nr, 11, bold=True, color=C_WHITE)

        stxb = slide.shapes.add_textbox(x + Inches(0.42), y, Inches(4.25), Inches(0.56))
        stf = stxb.text_frame
        stf.word_wrap = True
        sp = stf.paragraphs[0]
        sr = sp.add_run()
        sr.text = step
        _set_font(sr, 11, color=C_DARKTEXT)


# ── Main generator ────────────────────────────────────────────────────────────

def generate_investment_presentation() -> dict:
    """Build the PowerPoint presentation from pre-generated proposal JSON.

    Returns a metadata dict with slide_count, generated_charts, generated_tables,
    portfolio_used.
    """
    proposal_path = OUTPUT_DIRS["reports"] / "investment_proposal.json"
    mandate_path  = OUTPUT_DIRS["outputs"] / "context" / "investment_mandate.json"

    proposal  = json.loads(proposal_path.read_text(encoding="utf-8"))
    mandate   = json.loads(mandate_path.read_text(encoding="utf-8"))
    slides_data = proposal.get("slides", [])

    # Index slides by slide_number
    by_num = {s["slide_number"]: s for s in slides_data}

    # ── Pre-render charts ──────────────────────────────────────────────────────
    generated_charts = []

    # Asset allocation pie (slide 4)
    s4 = by_num.get(4, {}).get("content", {})
    pie_data = s4.get("pie_chart_data", [
        {"label": "Fixed Income", "value": s4.get("fixed_income_pct", 60)},
        {"label": "Equity",       "value": s4.get("equity_pct", 30)},
        {"label": "Liquidity",    "value": s4.get("liquidity_pct", 10)},
    ])
    asset_chart = _pie_to_image(
        [d["label"] for d in pie_data],
        [d["value"] for d in pie_data],
        "Asset Allocation"
    )
    generated_charts.append("Asset Allocation Pie Chart")

    # Currency allocation pie (slide 5)
    s5 = by_num.get(5, {}).get("content", {})
    ccy_data = s5.get("pie_chart_data", [
        {"label": "USD", "value": s5.get("usd_pct", 50)},
        {"label": "EUR", "value": s5.get("eur_pct", 30)},
        {"label": "RSD", "value": s5.get("rsd_pct", 20)},
    ])
    ccy_chart = _pie_to_image(
        [d["label"] for d in ccy_data],
        [d["value"] for d in ccy_data],
        "Currency Allocation"
    )
    generated_charts.append("Currency Allocation Pie Chart")

    # Sector allocation pie (slide 6)
    s6 = by_num.get(6, {}).get("content", {})
    sector_bd = s6.get("sector_breakdown", [])
    sector_chart = _pie_to_image(
        [s["sector"] for s in sector_bd] or ["—"],
        [s["pct"] for s in sector_bd] or [100],
        "Sector Allocation"
    )
    generated_charts.append("Sector Allocation Pie Chart")

    # Portfolio comparison bar (slide 9)
    s9 = by_num.get(9, {}).get("content", {})
    comparison = s9.get("comparison", [])

    def _parse_yield(y_str):
        import re
        m = re.search(r"[\d.]+", str(y_str))
        return float(m.group()) if m else 0.0

    port_names  = [c.get("portfolio", f"P{i+1}") for i, c in enumerate(comparison)]
    port_yields = [_parse_yield(c.get("expected_yield", "0")) for c in comparison]
    port_scores = [float(c.get("mandate_fit_score", 0)) for c in comparison]

    cmp_chart = _bar_to_image(
        port_names or ["Conservative", "Balanced", "Yield Focused"],
        {
            "Expected Yield (%)": port_yields or [4.5, 5.5, 6.5],
            "Mandate Fit Score": port_scores or [75, 85, 70],
        },
        "Portfolio Comparison"
    )
    generated_charts.append("Portfolio Comparison Bar Chart")

    # ── Build presentation ─────────────────────────────────────────────────────
    prs = Presentation()
    prs.slide_width  = Inches(10)
    prs.slide_height = Inches(5.625)
    prs.core_properties.title  = "Investment Proposal"
    prs.core_properties.author = "Investment Committee"

    generated_tables = []

    _slide_title(prs, mandate)                                      # 1

    _slide_exec_summary(prs, by_num.get(1, {}), 2)                  # 2

    _slide_market_env(prs, by_num.get(2, {}), 3)                    # 3

    _slide_recommended_portfolio(prs, by_num.get(3, {}), 4)         # 4

    _slide_asset_allocation(prs, by_num.get(4, {}), 5, asset_chart) # 5

    _slide_currency_allocation(prs, by_num.get(5, {}), 6, ccy_chart)# 6

    _slide_sector_allocation(prs, by_num.get(6, {}), 7, sector_chart) # 7
    generated_tables.append("Sector Breakdown Table")

    _slide_top_holdings(prs, by_num.get(7, {}), 8)                  # 8
    generated_tables.append("Top Holdings Table")

    _slide_portfolio_comparison(prs, by_num.get(9, {}), 9, cmp_chart) # 9
    generated_tables.append("Portfolio Comparison Table")

    _slide_risk_assessment(prs, by_num.get(8, {}), 10)              # 10

    _slide_expected_outcomes(prs, by_num.get(11, {}), 11)           # 11

    _slide_committee_recommendation(prs, by_num.get(10, {}), 12)    # 12

    _slide_next_steps(prs, by_num.get(12, {}), 13)                  # 13

    # Save pptx
    prs.save(str(PPTX_PATH))
    print(f"Saved PPTX: {PPTX_PATH}")

    # Attempt PDF conversion via LibreOffice
    pdf_generated = False
    try:
        import subprocess
        result = subprocess.run(
            ["libreoffice", "--headless", "--convert-to", "pdf",
             "--outdir", str(PRESENTATIONS_DIR), str(PPTX_PATH)],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode == 0 and PDF_PATH.exists():
            pdf_generated = True
            print(f"Saved PDF:  {PDF_PATH}")
        else:
            print(f"LibreOffice PDF conversion skipped (not available or failed).")
    except Exception:
        print("LibreOffice not found — PDF not generated.")

    # ── Determine portfolio used ───────────────────────────────────────────────
    recommended = (
        by_num.get(3, {}).get("content", {}).get("recommended_portfolio", "Unknown")
    )

    # ── Write metadata ─────────────────────────────────────────────────────────
    metadata = {
        "generated_at":    dt.datetime.utcnow().isoformat() + "Z",
        "slide_count":     TOTAL_SLIDES,
        "generated_charts": generated_charts,
        "generated_tables": generated_tables,
        "portfolio_used":  recommended,
        "pptx_path":       str(PPTX_PATH),
        "pdf_path":        str(PDF_PATH) if pdf_generated else None,
    }
    METADATA_PATH.write_text(json.dumps(metadata, indent=2), encoding="utf-8")
    print(f"Saved metadata: {METADATA_PATH}")

    return metadata


Note: you may need to restart the kernel to use updated packages.


Matplotlib is building the font cache; this may take a moment.


In [34]:
presentation = generate_investment_presentation()

print("Presentation generated successfully")
print(presentation)


Saved PPTX: /Users/ogurianova/Desktop/ANTON TASK1/outputs/presentations/investment_proposal.pptx
LibreOffice not found — PDF not generated.
Saved metadata: /Users/ogurianova/Desktop/ANTON TASK1/outputs/presentations/presentation_metadata.json
Presentation generated successfully
{'generated_at': '2026-06-12T14:04:33.693636Z', 'slide_count': 13, 'generated_charts': ['Asset Allocation Pie Chart', 'Currency Allocation Pie Chart', 'Sector Allocation Pie Chart', 'Portfolio Comparison Bar Chart'], 'generated_tables': ['Sector Breakdown Table', 'Top Holdings Table', 'Portfolio Comparison Table'], 'portfolio_used': 'Balanced (Portfolio B) — diversified HY/EM/IG fixed income + targeted equities in IT Services and Healthcare; liquidity buffer retained.', 'pptx_path': '/Users/ogurianova/Desktop/ANTON TASK1/outputs/presentations/investment_proposal.pptx', 'pdf_path': None}


/var/folders/tn/vwk_k1ld0777pqyjl__cd9b00000gn/T/ipykernel_6005/2280267850.py:739: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "generated_at":    dt.datetime.utcnow().isoformat() + "Z",
